# II-4 通信プロトコル実験（HTTP／MQTT／WebSocket）（ノートブック版）

教科書の II-4 のうち、**Pico・Pi・Wi-Fi 中継機がなくても動く部分**を1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約1分）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | 自作の WebSocket クライアントを RFC 6455 の例と小さなサーバーで検算する | 手順 1（mac_ii4.sh の 1)） | `pico/wsclient.py`・`test_wsclient.py` |
| 2 | 同じ計算機の中（ループバック）で、HTTP（1通ごとに接続）と WebSocket（つないだまま）の往復を比べる | ―（ノートブック版で追加） | `pico/wsclient.py` |
| 3 | 原理の数値を確かめる（ticks_us の一周、時計のずれ、バックオフ、$n_\text{eff}$、Wilson、多重比較） | II-4-3 | ― |
| 4 | 合成の記録を作る（実験 A・B・C の代わり） | 手順 D〜F の測定の代わり | `synth_ii4.py`（ノートブック版で追加） |
| 5 | 遅延・欠損の比較（時計合わせ、分位点、ブロック単位の回帰、Holm、統計の誤用） | 手順 12（実験 A） | `proto_analysis.py` |
| 6 | 消費電流の比較と1通あたりのエネルギー | 手順 15（実験 B） | `current_analysis.py` |
| 7 | 障害（中継機を抜く）の時間の流れ、欠損・重複、LWT | 手順 20（実験 C） | `fault_timeline.py` |

- 次のセルたちが、II-4-6 の `wsclient.py`・`test_wsclient.py` と、Mac の解析のファイル3つを、教科書と同じ並び（`pico/wsclient.py` など）でこのノートブックの隣に `%%writefile` で書き出します。中身は教科書と同じです。変えたのは、図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）だけです。
- `synth_ii4.py` は本文にはありません。本文の II-4-7 に「合成データで試したときの値」として書いてある値（HTTP の中央値 約21ms、ほかは約9〜11ms など）を出したときと同じ作り方で、`ii4_export.py` が書き出す `ii4_msgs.csv`、`mark` の `marks.csv`、`serial_logger.py` の電流の記録と同じ形のファイルを作ります（しくみはファイルの先頭の説明）。値はどれも仮のもので、実機の値ではありません。
- 2 節のループバックの比べは、Wi-Fi も TCP の再送もない「接続を作るコスト」だけの比べです。µs〜ms の値は計算機で大きく変わるので、`ok()` は「HTTP のほうが往復が長い」「WebSocket の見出しは 8 バイト」のような、計算機に依らないことだけを確かめます。MQTT はブローカ（Mosquitto）が要るので入れていません。
- 各節では、ターミナルの `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します。表示を見てから、`ok()` で教科書の「確かめること」「結果の例」を確かめます。
- **入れていないもの（実機が要る部分）**：Pi の受け口（`compose.yaml`・`api.py`・`pi_ii4.sh`・`ii4_export.py`。Docker・PostgreSQL・Mosquitto が要る）、Pico への書き込みと配線、`node.py`・`ina_meter.py` の実行、中継機を抜くなどの障害、ufw や合言葉（INGEST_TOKEN）の設定。Pico 側の中心の `node.py` と、受け口の `api.py` は、下に読むためだけに載せてあります（実行しません）。自分の `ii4_msgs.csv`・`marks.csv`・電流の記録を Colab の左のファイルの欄に上げれば、同じ `run("proto_analysis.py ii4_msgs.csv")` などで解析できるはずですが、執筆時には確かめていません。
- ⚠ 中継機は AC100V に直接挿す機器です。抜き挿しの注意と、家族への連絡、合言葉や Wi-Fi のパスワードを Git に入れないことは、教科書の II-4-4 を読んでください。このノートブックには秘密の値は1つも入っていません（`test_wsclient.py` の `good-token` はテスト用の見本の文字列です）。
- 乱数の種は固定してあるので、数値はほぼ同じになります（pandas・statsmodels の版で最後の桁が変わることがあります）。Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("statsmodels", "statsmodels")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, pandas, matplotlib, statsmodels
print("OK numpy", numpy.__version__, "scipy", scipy.__version__, "pandas", pandas.__version__,
      "matplotlib", matplotlib.__version__, "statsmodels", statsmodels.__version__)
print("（ノートブックは numpy 2.5.5・pandas 3.1.1・statsmodels 0.15.0 で確かめた）")
import os
os.makedirs("pico", exist_ok=True)
os.makedirs("data", exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

## Pico と Pi の側（読むだけ。実行しません）

教科書の手順では、Pico A・B の `node.py` が、乱数で決めた順番（plan）のブロックごとに方式（h = HTTP、m = MQTT QoS0、q = MQTT QoS1、w = WebSocket、i = 送らない）を切り替えながら、1秒に1通、シーケンス番号つきで送ります。送れなければバッファにためて、指数バックオフでつなぎ直します。Pi の `api.py` は HTTP の `POST /ingest` と WebSocket の `/ws` を受け、返事に受け取った時刻 t2・返す時刻 t3 を入れます。このノートブックの 4 節の `synth_ii4.py` は、これらが DB に残す記録と同じ形のものを作ります。

**Pico A・B（MicroPython）｜~/home-lab/partII/ii4/pico/node.py（Pico には :main.py）**（読むだけ）

```python
# II-4：同じデータを HTTP／MQTT（QoS 0・1）／WebSocket で送り比べる端末のプログラム（Pico A・B 共通、MicroPython）
# 置き場所: Mac の ~/home-lab/partII/ii4/pico/node.py → Pico には :main.py としてコピー
# 準備（Pico ごとに1回）:
#   mpremote connect auto mip install umqtt.simple          # 第0章で入れていれば不要
#   mpremote connect auto exec "import requests"            # エラーなら: mpremote connect auto mip install requests
#   mpremote connect auto fs cp ../../lib/pico/bme280.py :lib/bme280.py
#   mpremote connect auto fs cp ../../lib/pico/ina219.py :lib/ina219.py
#   mpremote connect auto fs cp pico/wsclient.py :wsclient.py
#   mpremote connect auto fs cp pico/labconfig_a.py :labconfig.py      # Pico B は labconfig_b.py
#   mpremote connect auto fs cp pico/node.py :main.py
# 動き:
#   電源を入れると Wi-Fi につなぎ、ブロックの順番（PLAN）を乱数で決めて、1秒に1通ずつ送る。
#   1ブロック = BLOCK_N 通で、ブロックごとに方式を切り替える（h=HTTP、m=MQTT QoS0、q=MQTT QoS1、w=WebSocket、
#   i=何も送らない＝電流の基準）。送れなかった通は端末のバッファにためて、つながり直したら古い順に送る。
#   つなぎ直しは指数バックオフ（0.5→1→2→…→最大30秒、ゆらぎつき）。
#   ブロックが切り替わったとき、前の方式で送れずに残った通は捨てて ns に数える（その方式の欠損として数えるため）。
import gc
import json
import os
import time

import machine
import network
import rp2
from machine import I2C, Pin

import labconfig as cfg
import wsclient
from umqtt.simple import MQTTClient

try:
    import requests
except ImportError:
    import urequests as requests

# ===== 実験の設定（Pico A・B で同じにする。実験B では Pico B だけ ARMS = "hmqwi"）=====
ARMS = "hmqw"            # 比べる方式。i を入れると「Wi-Fi にはつないだまま何も送らない」ブロックができる
BLOCKS_PER_ARM = 4       # 方式ごとのブロック数（4方式 × 4 = 16ブロック ≒ 16分）
BLOCK_N = 60             # 1ブロックの通数
PERIOD_MS = 1000         # 送る間隔
FIXED = None             # 障害の実験：("q", 600) のように書くと、その方式だけで 600 通送る
TIMEOUT_S = 3            # 1回の送信・接続の待ち時間の上限
BACKOFF_MIN_S = 0.5      # つなぎ直しの待ち時間の最初の値
BACKOFF_MAX_S = 30       # 待ち時間の上限
BUF_MAX = 240            # 端末にためておける通数（あふれたら古いものから捨てて nd を数える）
SEND_BURST = 5           # 1回のループで送る最大の通数（たまった分を吐き出すとき）
MQTT_KEEPALIVE_S = 10    # ブローカはこの 1.5 倍の間なにも届かなければ切断とみなし、LWT を出す
MQTT_PING_S = 5          # 送るものが無いとき、この間隔で PINGREQ を送って生きているか確かめる
WIFI_PM = "none"         # "none" = 省電力を切る（第0章と同じ）、"default" = ファームの既定のまま
FLUSH_S = 60             # 全部作り終えたあと、残りを送りきるまで待つ時間の上限
START_DELAY_S = 15       # Wi-Fi につないでから送り始めるまで待つ時間（電流の記録などを先に始めておくため）

DEV = cfg.DEVICE_ID + "-ii4"
HOST = cfg.MQTT_HOST
TOPIC = ("lab/%s/telemetry" % DEV).encode()
STATUS = ("lab/%s/status" % DEV).encode()
URL = "http://%s:%d/ingest" % (HOST, cfg.API_PORT)
HDR = {"Content-Type": "application/json", "X-Lab-Token": cfg.API_TOKEN}

rp2.country("JP")
led = Pin("LED", Pin.OUT)
wlan = network.WLAN(network.STA_IF)


class Clock:
    """ticks_us は約17.9分（2の30乗µs）で一周するので、差を足し続けて、増え続ける µs にする"""
    def __init__(self):
        self.last = time.ticks_us()
        self.acc = 0

    def us(self):
        now = time.ticks_us()
        self.acc += time.ticks_diff(now, self.last)   # 呼ぶ間隔が約9分（半周）より短ければ正しい
        self.last = now
        return self.acc


clock = Clock()


def rand01():
    return int.from_bytes(os.urandom(2), "big") / 65535


def make_plan():
    if FIXED:
        return FIXED[0], FIXED[1]
    p = list(ARMS * BLOCKS_PER_ARM)
    for i in range(len(p) - 1, 0, -1):                 # Fisher–Yates のシャッフル
        j = int.from_bytes(os.urandom(2), "big") % (i + 1)
        p[i], p[j] = p[j], p[i]
    return "".join(p), BLOCK_N


# ===== センサ（入門3 のドライバ）=====
i2c = I2C(0, sda=Pin(4), scl=Pin(5), freq=100_000)
if cfg.SENSOR == "bme":
    from bme280 import BME280
    bme = BME280(i2c)
else:
    from ina219 import INA219
    ina = INA219(i2c)                                  # Pico B の消費電流（VBUS → INA219 → Pico B の VSYS）
isum, imax, icnt = 0.0, -1e9, 0


def sample_current():
    """Pico A：ループのたびに電流を読んでためる（送っている間は読めないので、送る値は参考値）"""
    global isum, imax, icnt
    i = ina.shunt_mv() / ina.r_shunt
    isum += i
    icnt += 1
    if i > imax:
        imax = i


def read_sensor():
    global isum, imax, icnt
    if cfg.SENSOR == "bme":
        t, p, h = bme.read()
        return {"temp_c": round(t, 2), "press_pa": round(p, 1), "hum_pct": None if h is None else round(h, 2)}
    d = {"bus_v": round(ina.bus_v(), 3), "i_ma": round(isum / icnt, 2) if icnt else None,
         "i_max_ma": round(imax, 2) if icnt else None, "n_i": icnt}
    isum, imax, icnt = 0.0, -1e9, 0
    return d


def rssi():
    try:
        return wlan.status("rssi")
    except Exception:
        return None


def status_msg(state, run):
    return json.dumps({"dev": DEV, "run": run, "state": state})


# ===== 送り方（どれも send(body, seq) → (かかった時間 µs, 時計合わせの4つの時刻 または None)）=====
class HttpTr:
    arm = "h"

    def send(self, body, seq):
        t1 = clock.us()
        r = requests.post(URL, data=body.encode(), headers=HDR, timeout=TIMEOUT_S)   # 1通ごとに TCP をつなぐ
        try:
            if r.status_code != 200:
                raise OSError("HTTP %d" % r.status_code)
            j = r.json()
        finally:
            r.close()
        t4 = clock.us()
        # 時計合わせには使わない：t1 は TCP の接続の前なので、行きに接続の往復が余計に入ってかたよる
        return t4 - t1, None

    def alive(self):
        pass

    def close(self):
        pass


class TSock:
    """umqtt.simple は受信のたびに setblocking(True) にして待ち時間の上限を消してしまうので、
    読み書きのたびに上限をかけ直す包み（umqtt.simple の中身に依存した工夫。版が変わったら要確認）"""
    def __init__(self, s, t):
        self.s, self.t = s, t
        s.settimeout(t)

    def read(self, n):
        self.s.settimeout(self.t)
        return self.s.read(n)

    def write(self, *a):
        self.s.settimeout(self.t)
        return self.s.write(*a)

    def setblocking(self, flag):
        pass

    def settimeout(self, t):
        pass

    def close(self):
        self.s.close()


class MqttTr:
    def __init__(self, qos, run):
        self.qos, self.run = qos, run
        self.arm = "m" if qos == 0 else "q"
        self.c = MQTTClient(("pico-%s-%s" % (DEV, self.arm)).encode(), HOST, port=getattr(cfg, "MQTT_PORT", 1883),
                            user=cfg.MQTT_USER, password=cfg.MQTT_PASS, keepalive=MQTT_KEEPALIVE_S)
        # LWT（遺言）：切断に気づいたブローカが代わりに出す。retain なので、あとから購読した人にも最後の状態が分かる
        self.c.set_last_will(STATUS, status_msg("lwt", run), retain=True, qos=1)
        try:
            try:
                self.c.connect(timeout=TIMEOUT_S)
            except TypeError:                             # 古い umqtt.simple は timeout を受け付けない
                self.c.connect()
            self.c.sock = TSock(self.c.sock, TIMEOUT_S)
            self.c.publish(STATUS, status_msg("online", run), retain=True, qos=1)
        except Exception:
            try:                                          # 失敗したらソケットを閉じる（閉じないと lwIP の資源を使い切る）
                if self.c.sock:
                    self.c.sock.close()
            except Exception:
                pass
            raise
        self.last_ping = clock.us()

    def send(self, body, seq):
        t1 = clock.us()
        self.c.publish(TOPIC, body.encode(), qos=self.qos)   # QoS1 は PUBACK が返るまで待つ
        t4 = clock.us()
        self.last_ping = t4
        return t4 - t1, None                              # QoS0 の値は「書き込みにかかった時間」で往復ではない

    def alive(self):
        if clock.us() - self.last_ping < MQTT_PING_S * 1_000_000:
            return
        self.c.ping()
        for _ in range(3):
            if self.c.wait_msg() is None:                 # PINGRESP が来ると None
                break
        self.last_ping = clock.us()

    def close(self):
        try:
            self.c.publish(STATUS, status_msg("offline", self.run), retain=True, qos=0)
            self.c.disconnect()                           # 正しく切ると LWT は出ない
        except Exception:
            pass
        try:
            self.c.sock.close()
        except Exception:
            pass


class WsTr:
    arm = "w"

    def __init__(self):
        self.ws = wsclient.connect(HOST, cfg.API_PORT, "/ws?dev=" + DEV,
                                   headers={"X-Lab-Token": cfg.API_TOKEN}, timeout=TIMEOUT_S)

    def send(self, body, seq):
        t1 = clock.us()
        self.ws.send_text(body)
        while True:                                       # 自分の seq の ack が来るまで読む（ping には自動で pong）
            j = json.loads(self.ws.recv())
            if j.get("ack") == seq:
                break
        t4 = clock.us()
        return t4 - t1, (t1, t4, j["t2"], j["t3"])

    def alive(self):
        pass

    def close(self):
        try:
            self.ws.close(1000)
        except Exception:
            pass


def open_tr(arm, run):
    if arm == "h":
        return HttpTr()
    if arm == "m":
        return MqttTr(0, run)
    if arm == "q":
        return MqttTr(1, run)
    return WsTr()


def wifi_setup():
    wlan.active(True)
    if WIFI_PM == "none":
        try:
            wlan.config(pm=network.WLAN.PM_NONE)       # 名前はファームの版で要確認
        except Exception as e:
            print("I,pm の設定に失敗:", e)
    wlan.connect(cfg.WIFI_SSID, cfg.WIFI_PASS)
    t0 = time.ticks_ms()
    while not wlan.isconnected() and time.ticks_diff(time.ticks_ms(), t0) < 30_000:
        time.sleep_ms(250)
    print("I,Wi-Fi", wlan.isconnected(), wlan.ifconfig()[0], "rssi", rssi())
    time.sleep(START_DELAY_S)


def main():
    wifi_setup()
    run = "%08x" % int.from_bytes(os.urandom(4), "big")
    plan, bn = make_plan()
    total = len(plan) * bn
    period = PERIOD_MS * 1000
    print("I,dev=%s run=%s plan=%s bn=%d" % (DEV, run, plan, bn))
    buf = []                     # [seq, 送った回数, 作ったときの JSON, 方式]
    seq = 0
    tr = None
    backoff = BACKOFF_MIN_S
    next_try = 0
    wifi_next = 0
    last_rtt, last_sync = None, None
    nf = nc = nd = ns = 0        # 失敗の回数、接続した回数、バッファからあふれて捨てた数、切り替えで捨てた数
    prev = plan[0]
    t_start = clock.us()
    next_gen = t_start
    t_done = None
    while True:
        now = clock.us()
        # 1) 予定の時刻になったら1通作る（時刻 tg は「予定の時刻」。遅れて作っても予定の時刻を入れる）
        if seq < total and now >= next_gen:
            blk, k = divmod(seq, bn)
            arm = plan[blk]
            if k == 0:
                print("I,block %d/%d arm=%s buf=%d nf=%d nc=%d nd=%d ns=%d" % (blk + 1, len(plan), arm, len(buf), nf, nc, nd, ns))
                gc.collect()
            base = {"dev": DEV, "run": run, "plan": plan, "bn": bn, "per": PERIOD_MS, "seq": seq, "blk": blk,
                    "k": k, "arm": arm, "tg": next_gen, "rssi": rssi()}
            base.update(read_sensor())
            if arm != "i":
                buf.append([seq, 0, json.dumps(base), arm])
                if len(buf) > BUF_MAX:
                    buf.pop(0)
                    nd += 1
            seq += 1
            next_gen += period
        elif cfg.SENSOR == "ina":
            sample_current()
        cur = plan[(seq - 1) // bn] if seq else plan[0]
        # 2) 方式が変わったら、前の方式で送れなかった通を捨てて数え（ns）、今の接続を閉じる
        #    残りを次の方式で送ると、遅延や欠損が別の方式のものとして数えられてしまうため
        if cur != prev:
            k0 = len(buf)
            buf = [b for b in buf if b[3] == cur]
            ns += k0 - len(buf)
            prev = cur
        if tr is not None and tr.arm != cur:
            tr.close()
            tr = None
            backoff, next_try = BACKOFF_MIN_S, now
        # 3) Wi-Fi が切れていたら、15秒ごとにつなぎ直しを頼む（待たずに先へ進む）
        if not wlan.isconnected():
            if tr is not None:
                tr.close()
                tr = None
            if now >= wifi_next:
                try:
                    wlan.connect(cfg.WIFI_SSID, cfg.WIFI_PASS)
                except OSError:
                    pass
                wifi_next = now + 15_000_000
        # 4) つながっていなければ、待ち時間が過ぎたらつなぐ。つながっていれば、たまった分を古い順に送る
        elif cur != "i" and (buf or tr is not None):
            try:
                if tr is None and buf and now >= next_try:
                    tr = open_tr(cur, run)
                    nc += 1
                if tr is not None:
                    sent = 0
                    while buf and sent < SEND_BURST:
                        item = buf[0]
                        item[1] += 1
                        extra = {"ts": clock.us(), "try": item[1], "rtt": last_rtt, "nf": nf, "nc": nc, "nd": nd,
                                 "ns": ns, "buf": len(buf)}
                        if last_sync:
                            extra["x1"], extra["x4"], extra["x2"], extra["x3"] = last_sync
                        body = json.dumps(extra)[:-1] + ", " + item[2][1:]     # 2つの JSON の中身をつなぐ
                        last_rtt, sync = tr.send(body, item[0])
                        if sync:
                            last_sync = sync
                        buf.pop(0)
                        sent += 1
                        backoff = BACKOFF_MIN_S                # 送れたら待ち時間を最初に戻す
                        led.toggle()
                    if not buf:
                        tr.alive()
            except Exception as e:
                nf += 1
                print("I,fail arm=%s %s: %s buf=%d next=%.1fs" % (cur, type(e).__name__, e, len(buf), backoff))
                if tr is not None:
                    tr.close()
                tr = None
                # 指数バックオフ（半分は固定、半分は乱数）：みんなが同時につなぎ直して押し寄せるのを防ぐ
                next_try = clock.us() + int((backoff / 2 + backoff / 2 * rand01()) * 1_000_000)
                backoff = min(backoff * 2, BACKOFF_MAX_S)
        # 5) 終わり：全部作り、送りきった（または FLUSH_S を過ぎた）ら閉じる
        if seq >= total:
            if t_done is None:
                t_done = now
            if not buf or now - t_done > FLUSH_S * 1_000_000:
                if tr is not None:
                    tr.close()
                print("I,done run=%s sent=%d left=%d nf=%d nc=%d nd=%d ns=%d" % (run, total, len(buf), nf, nc, nd, ns))
                led.on()
                return
        time.sleep_ms(2 if cfg.SENSOR == "ina" else 5)


main()
```

**Mac → Pi｜stack/app/api.py（II-4 の版）**（読むだけ）

```python
"""II-4：データ受信 API（FastAPI）。HTTP と WebSocket で JSON を受け取り、PostgreSQL に入れる
（II-3 の api.py を置き換える）

  POST /ingest            … HTTP。1回に1通。ヘッダ X-Lab-Token が必要。トピックは http/<dev>
  WS   /ws?dev=<dev>      … WebSocket。つないだまま何通でも送れる。ハンドシェイクに X-Lab-Token が必要。トピックは ws/<dev>
  GET  /health            … DB につながるか（エラーの中身は返さない）
  GET  /latest            … ?dev=pico1&n=5 で最新の n 件。LAN に開いたので、これにも X-Lab-Token が必要
返事には t2（受け取った時刻）と t3（返事を出す時刻）を、UNIX 時刻の「整数のマイクロ秒」で入れる。
Pico は送った時刻 t1 と返事を受け取った時刻 t4 を自分の時計で測り、時計のずれの推定に使う（II-4-3-3）。
Pico の MicroPython の float は精度が足りないことがあるので、時刻は整数で返す。
"""
import hmac
import json
import os
import threading
import time

import psycopg
from fastapi import FastAPI, Header, HTTPException, WebSocket, WebSocketDisconnect
from pydantic import BaseModel, ConfigDict
from starlette.concurrency import run_in_threadpool

app = FastAPI(title="home-lab ingest")
TOKEN = os.environ.get("INGEST_TOKEN", "")       # .env の INGEST_TOKEN。空なら全部断る（閉じた側に倒す）

_lock = threading.Lock()
_con = None


class Msg(BaseModel):
    model_config = ConfigDict(extra="allow")     # dev・seq 以外の項目もそのまま受け取る
    dev: str
    seq: int | None = None


def now_us():
    return time.time_ns() // 1000


def token_ok(tok):
    return bool(TOKEN) and tok is not None and hmac.compare_digest(tok.encode(), TOKEN.encode())


def store(t_us, topic, dev, seq, body):
    """1行を書き込む。接続は1本を使い回し（毎回つなぐと数ms 余計にかかる）、切れていたらつなぎ直す"""
    global _con
    with _lock:
        for attempt in (1, 2):
            try:
                if _con is None or _con.closed:
                    _con = psycopg.connect(connect_timeout=3, autocommit=True)   # 接続先は環境変数 PGHOST など
                _con.execute("INSERT INTO messages (recv_unix, topic, dev, seq, payload) VALUES (%s, %s, %s, %s, %s)",
                             (t_us / 1e6, topic, dev, seq, json.dumps(body)))
                return
            except psycopg.OperationalError:
                _con = None
                if attempt == 2:
                    raise


@app.get("/health")
def health():
    try:
        with psycopg.connect(connect_timeout=3) as c:
            c.execute("SELECT 1")
        return {"ok": True}
    except Exception:
        raise HTTPException(status_code=503, detail="db unavailable")    # 中身（接続先など）は外に見せない


@app.post("/ingest")
def ingest(m: Msg, x_lab_token: str | None = Header(default=None)):
    t2 = now_us()
    if not token_ok(x_lab_token):
        raise HTTPException(status_code=401, detail="bad token")
    body = m.model_dump()
    store(t2, f"http/{m.dev}", m.dev, m.seq, body)
    return {"ok": True, "seq": m.seq, "recv_unix": t2 / 1e6, "t2": t2, "t3": now_us()}


@app.websocket("/ws")
async def ws_ingest(ws: WebSocket, dev: str = "unknown"):
    if not token_ok(ws.headers.get("x-lab-token")):
        await ws.close(code=1008)                 # 1008 = 決まりに反する（ハンドシェイクの前なら HTTP 403 で断られる）
        return
    await ws.accept()
    try:
        while True:
            text = await ws.receive_text()        # ping・pong・close の制御フレームは uvicorn が処理する
            t2 = now_us()
            try:
                d = json.loads(text)
                if not isinstance(d, dict):
                    d = {"value": d}
            except json.JSONDecodeError:
                d = {"raw": text}
            seq = d.get("seq") if isinstance(d.get("seq"), int) else None
            await run_in_threadpool(store, t2, f"ws/{d.get('dev', dev)}", d.get("dev", dev), seq, d)
            await ws.send_text(json.dumps({"ack": seq, "t2": t2, "t3": now_us()}))
    except WebSocketDisconnect:
        pass


@app.get("/latest")
def latest(dev: str, n: int = 5, x_lab_token: str | None = Header(default=None)):
    if not token_ok(x_lab_token):
        raise HTTPException(status_code=401, detail="bad token")
    with psycopg.connect(connect_timeout=3) as c:
        rows = c.execute("SELECT recv_time, seq, payload FROM messages WHERE dev = %s ORDER BY id DESC LIMIT %s",
                         (dev, min(n, 100))).fetchall()
    return [{"recv_time": r[0].isoformat(), "seq": r[1], "payload": r[2]} for r in rows]
```

次のセルたちで、WebSocket クライアント（Pico と Mac の CPython の両方で動く）とその検算、Mac の解析のファイル3つ、合成の記録を作る `synth_ii4.py`（ノートブック版で追加）を書き出します。

In [ ]:
%%writefile pico/wsclient.py
# II-4：最小限の WebSocket クライアント（RFC 6455）。MicroPython と CPython の両方で動く
# 置き場所: Mac の ~/home-lab/partII/ii4/pico/wsclient.py → Pico には :wsclient.py としてコピー
#           Mac の test_wsclient.py が、同じファイルを CPython で検算する
# できること: ハンドシェイク（Sec-WebSocket-Accept の確認）、マスク付きのテキストフレームの送信、
#             受信（分割されたフレームの組み立て）、ping への pong、close の手順
# できないこと: TLS（wss://）、拡張（圧縮など）、サブプロトコル
import os
import socket
import struct
import binascii
import hashlib

GUID = b"258EAFA5-E914-47DA-95CA-C5AB0DC85B11"      # RFC 6455 で決まっている固定の文字列
OP_CONT, OP_TEXT, OP_BIN, OP_CLOSE, OP_PING, OP_PONG = 0x0, 0x1, 0x2, 0x8, 0x9, 0xA


class WSClosed(Exception):
    """相手から close を受け取った、または手順の誤りで閉じた。args = (コード, 理由)"""


def accept_key(key):
    """クライアントの Sec-WebSocket-Key（base64 の bytes）から、サーバーが返すはずの Accept を計算する"""
    sha1 = getattr(hashlib, "sha1", None)
    if sha1 is None:                                  # ファームによっては sha1 が無い（その場合は確認を省く）
        return None
    return binascii.b2a_base64(sha1(key + GUID).digest()).strip()


def build_frame(opcode, payload, mask_key=None, fin=True):
    """クライアント→サーバーのフレーム。クライアントは必ずマスクする（RFC 6455 5.3）"""
    if isinstance(payload, str):
        payload = payload.encode()
    if mask_key is None:
        mask_key = os.urandom(4)
    n = len(payload)
    b0 = (0x80 if fin else 0) | opcode
    if n < 126:
        hdr = struct.pack("!BB", b0, 0x80 | n)
    elif n < 65536:
        hdr = struct.pack("!BBH", b0, 0x80 | 126, n)
    else:
        hdr = struct.pack("!BBQ", b0, 0x80 | 127, n)
    body = bytearray(payload)
    for i in range(n):
        body[i] ^= mask_key[i & 3]
    return hdr + bytes(mask_key) + bytes(body)


class WebSocket:
    def __init__(self, sock):
        self.sock = sock
        self.closed = False
        self._frag = None                             # 分割されたメッセージの途中（[opcode, bytearray]）

    # --- 下まわり ---
    def _send(self, data):
        if hasattr(self.sock, "sendall"):             # CPython
            self.sock.sendall(data)
            return
        mv = memoryview(data)                         # MicroPython：送れた分だけ進める
        while len(mv):
            k = self.sock.send(mv)
            mv = mv[k:]

    def _read_exact(self, n):
        buf = b""
        while len(buf) < n:
            chunk = self.sock.recv(n - len(buf))
            if not chunk:
                raise OSError("connection closed by peer")
            buf += chunk
        return buf

    def _fail(self, code, why):
        """手順の誤りを見つけたら、close を送ってすぐ閉じる（RFC 6455 7.1.7）"""
        try:
            if not self.closed:
                self._send(build_frame(OP_CLOSE, struct.pack("!H", code)))
        except OSError:
            pass
        self.closed = True
        self.sock.close()
        raise WSClosed(code, why)

    # --- 送る ---
    def send_text(self, text):
        self._send(build_frame(OP_TEXT, text))

    def ping(self, data=b""):
        self._send(build_frame(OP_PING, data))

    # --- 受け取る ---
    def recv(self):
        """データのメッセージ（str か bytes）を1つ返す。ping には pong を返し、pong は読み捨てる"""
        while True:
            b0, b1 = self._read_exact(2)
            fin, rsv, op = b0 & 0x80, b0 & 0x70, b0 & 0x0F
            masked, n = b1 & 0x80, b1 & 0x7F
            if rsv:
                self._fail(1002, "RSV bits set")      # 拡張を決めていないのに RSV が立っている
            if n == 126:
                n = struct.unpack("!H", self._read_exact(2))[0]
            elif n == 127:
                n = struct.unpack("!Q", self._read_exact(8))[0]
            if masked:
                self._fail(1002, "server frame must not be masked")   # サーバー→クライアントはマスクしない決まり
            payload = self._read_exact(n) if n else b""
            if op >= 0x8:                             # 制御フレーム：分割されず、125バイト以下
                if not fin or n > 125:
                    self._fail(1002, "bad control frame")
                if op == OP_PING:
                    self._send(build_frame(OP_PONG, payload))   # 同じ中身で pong を返す
                    continue
                if op == OP_PONG:
                    continue
                if op == OP_CLOSE:
                    code = struct.unpack("!H", payload[:2])[0] if n >= 2 else 1005
                    reason = payload[2:].decode() if n > 2 else ""
                    if not self.closed:               # 相手からの close には close を返してから閉じる
                        try:
                            self._send(build_frame(OP_CLOSE, payload[:2] if n >= 2 else b""))
                        except OSError:
                            pass
                    self.closed = True
                    self.sock.close()
                    raise WSClosed(code, reason)
                self._fail(1002, "unknown control opcode")
            if op in (OP_TEXT, OP_BIN):
                if self._frag is not None:
                    self._fail(1002, "new message inside fragmented message")
                if fin:
                    return payload.decode() if op == OP_TEXT else payload
                self._frag = [op, bytearray(payload)]
            elif op == OP_CONT:
                if self._frag is None:
                    self._fail(1002, "continuation without start")
                self._frag[1] += payload
                if fin:
                    op0, data = self._frag
                    self._frag = None
                    return bytes(data).decode() if op0 == OP_TEXT else bytes(data)
            else:
                self._fail(1002, "unknown opcode")

    # --- 閉じる ---
    def close(self, code=1000):
        """close を送り、相手の close を少し待ってから閉じる"""
        if self.closed:
            return
        try:
            self._send(build_frame(OP_CLOSE, struct.pack("!H", code)))
            self.closed = True
            for _ in range(5):                        # 相手の close が来るまで、届いた分を読み捨てる
                self.recv()
        except (OSError, WSClosed):
            pass
        finally:
            self.closed = True
            self.sock.close()


def connect(host, port, path="/", headers=None, timeout=5):
    """ws://host:port/path につなぐ。headers は追加のヘッダ（dict）。timeout 秒でタイムアウト"""
    ai = socket.getaddrinfo(host, port, 0, socket.SOCK_STREAM)[0]
    s = socket.socket(ai[0], socket.SOCK_STREAM, ai[2])
    s.settimeout(timeout)
    try:
        s.connect(ai[-1])
        key = binascii.b2a_base64(os.urandom(16)).strip()            # 16バイトの乱数を base64 にしたもの
        req = ("GET %s HTTP/1.1\r\nHost: %s:%d\r\nUpgrade: websocket\r\nConnection: Upgrade\r\n"
               "Sec-WebSocket-Key: %s\r\nSec-WebSocket-Version: 13\r\n" % (path, host, port, key.decode()))
        for k in (headers or {}):
            req += "%s: %s\r\n" % (k, headers[k])
        ws = WebSocket(s)
        ws._send((req + "\r\n").encode())
        resp = b""
        while not resp.endswith(b"\r\n\r\n"):            # 1バイトずつ読む（続くフレームを読みすぎないため）
            c = s.recv(1)
            if not c:
                raise OSError("connection closed during handshake")
            resp += c
            if len(resp) > 4096:
                raise OSError("handshake response too long")
        lines = resp.decode().split("\r\n")
        status = lines[0].split(" ")
        if len(status) < 2 or status[1] != "101":
            raise OSError("handshake failed: " + lines[0])
        h = {}
        for ln in lines[1:]:
            if ":" in ln:
                k, v = ln.split(":", 1)
                h[k.strip().lower()] = v.strip()
        if h.get("upgrade", "").lower() != "websocket" or "upgrade" not in h.get("connection", "").lower():
            raise OSError("not a websocket upgrade")
        exp = accept_key(key)
        if exp is not None and h.get("sec-websocket-accept", "").encode() != exp:
            raise OSError("bad Sec-WebSocket-Accept")
        return ws
    except Exception:
        s.close()
        raise

In [ ]:
%%writefile test_wsclient.py
#!/usr/bin/env python3
"""II-4：pico/wsclient.py を Mac の CPython で検算する（Pico も Pi もいらない）

使い方: cd ~/home-lab/partII/ii4 && uv run python test_wsclient.py
  1. RFC 6455 の例（Sec-WebSocket-Accept、マスク付きの "Hello"、長さの書き方）と一致するか
  2. サーバー役のフレームを流し込み、分割メッセージ・ping→pong・close の扱いが決まりどおりか
  3. このファイルの中の小さなサーバー（別スレッド）に実際につなぎ、送って返事を受け取れるか
"""
import pathlib
import socket
import struct
import sys
import threading

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parent / "pico"))
import wsclient as wc  # noqa: E402


def unmask(frame):
    """クライアントのフレームを、サーバーの立場で読む → (fin, opcode, 中身)"""
    b0, b1 = frame[0], frame[1]
    assert b1 & 0x80, "クライアントのフレームはマスクされていなければならない"
    n, i = b1 & 0x7F, 2
    if n == 126:
        n, i = struct.unpack("!H", frame[2:4])[0], 4
    elif n == 127:
        n, i = struct.unpack("!Q", frame[2:10])[0], 10
    key, data = frame[i:i + 4], frame[i + 4:i + 4 + n]
    return b0 & 0x80, b0 & 0x0F, bytes(c ^ key[j & 3] for j, c in enumerate(data))


def read_frame(sock):
    hdr = sock.recv(2)
    n = hdr[1] & 0x7F
    ext = sock.recv(2) if n == 126 else sock.recv(8) if n == 127 else b""
    if ext:
        n = int.from_bytes(ext, "big")
    rest = b""
    while len(rest) < 4 + n:
        rest += sock.recv(4 + n - len(rest))
    return unmask(hdr + ext + rest)


def test_vectors():
    # RFC 6455 1.3 の例
    assert wc.accept_key(b"dGhlIHNhbXBsZSBub25jZQ==") == b"s3pPLMBiTxaQ9kYGzzhZRbK+xOo="
    # RFC 6455 5.7 の例：マスク付きの "Hello"（マスクの鍵 37 fa 21 3d）
    f = wc.build_frame(wc.OP_TEXT, "Hello", mask_key=bytes([0x37, 0xFA, 0x21, 0x3D]))
    assert f == bytes.fromhex("818537fa213d7f9f4d5158"), f.hex()
    # 長さの書き方：125 までは1バイト、126〜65535 は 126＋2バイト、それより長いと 127＋8バイト
    assert wc.build_frame(wc.OP_BIN, b"x" * 125)[:2] == bytes([0x82, 0x80 | 125])
    assert wc.build_frame(wc.OP_BIN, b"x" * 256)[:4] == bytes([0x82, 0xFE, 0x01, 0x00])
    assert wc.build_frame(wc.OP_BIN, b"x" * 65536)[:10] == bytes([0x82, 0xFF, 0, 0, 0, 0, 0, 1, 0, 0])
    for n in (0, 1, 125, 126, 65535, 65536):
        assert unmask(wc.build_frame(wc.OP_BIN, bytes(range(256)) * (n // 256) + bytes(n % 256))) [2] == \
            bytes(range(256)) * (n // 256) + bytes(n % 256)
    print("1. RFC 6455 の例と一致: OK")


def test_frames():
    a, b = socket.socketpair()                        # a = クライアント側、b = サーバー役
    a.settimeout(2)
    b.settimeout(2)
    ws = wc.WebSocket(a)
    # 分割された "Hello"（RFC 6455 5.7 の例）の間に ping をはさむ（制御フレームは途中に入ってよい）
    b.sendall(bytes.fromhex("010348656c") + bytes.fromhex("890548656c6c6f") + bytes.fromhex("80026c6f"))
    assert ws.recv() == "Hello"
    fin, op, data = read_frame(b)
    assert (op, data) == (wc.OP_PONG, b"Hello"), "ping と同じ中身の pong を返す"
    # 126バイト以上の長さ（2バイトの拡張長）
    b.sendall(bytes([0x81, 126]) + struct.pack("!H", 300) + b"a" * 300)
    assert ws.recv() == "a" * 300
    # サーバーからの close（1000）→ 同じコードの close を返して WSClosed
    b.sendall(bytes([0x88, 0x02]) + struct.pack("!H", 1000))
    try:
        ws.recv()
        raise AssertionError("WSClosed が出るはず")
    except wc.WSClosed as e:
        assert e.args[0] == 1000
    fin, op, data = read_frame(b)
    assert op == wc.OP_CLOSE and struct.unpack("!H", data)[0] == 1000
    # マスクされたサーバーのフレームは決まり違反 → 1002 で閉じる
    a, b = socket.socketpair()
    ws = wc.WebSocket(a)
    b.sendall(bytes([0x81, 0x85]) + b"\x00\x00\x00\x00Hello")
    try:
        ws.recv()
        raise AssertionError("WSClosed が出るはず")
    except wc.WSClosed as e:
        assert e.args[0] == 1002
    print("2. 分割・ping→pong・close・決まり違反の扱い: OK")


def mini_server(srv, token, log):
    """テスト用の小さなサーバー：ハンドシェイク → ping を1回送る → 届いたテキストに ack を返す → close"""
    conn, _ = srv.accept()
    conn.settimeout(3)
    req = b""
    while not req.endswith(b"\r\n\r\n"):
        req += conn.recv(1)
    h = {k.strip().lower(): v.strip() for k, v in
         (ln.split(":", 1) for ln in req.decode().split("\r\n")[1:] if ":" in ln)}
    if h.get("x-lab-token") != token:
        conn.sendall(b"HTTP/1.1 403 Forbidden\r\nContent-Length: 0\r\n\r\n")
        conn.close()
        return
    acc = wc.accept_key(h["sec-websocket-key"].encode()).decode()
    conn.sendall(("HTTP/1.1 101 Switching Protocols\r\nUpgrade: websocket\r\nConnection: Upgrade\r\n"
                  "Sec-WebSocket-Accept: %s\r\n\r\n" % acc).encode())
    conn.sendall(bytes([0x89, 0x04]) + b"beat")       # サーバーからの ping
    for _ in range(3):
        fin, op, data = read_frame(conn)
        log.append((op, data))
        if op == wc.OP_TEXT:
            reply = b'{"ack": 1}'
            conn.sendall(bytes([0x81, len(reply)]) + reply)
        elif op == wc.OP_CLOSE:
            conn.sendall(bytes([0x88, 0x02]) + data[:2])
            break
    conn.close()


def test_live():
    for token, ok in (("good-token", True), ("bad-token", False)):
        srv = socket.socket()
        srv.bind(("127.0.0.1", 0))
        srv.listen(1)
        port = srv.getsockname()[1]
        log = []
        th = threading.Thread(target=mini_server, args=(srv, "good-token", log), daemon=True)
        th.start()
        try:
            ws = wc.connect("127.0.0.1", port, "/ws?dev=test", headers={"X-Lab-Token": token}, timeout=3)
            assert ok, "合言葉が違うのにつながった"
            ws.send_text('{"dev": "test", "seq": 1}')
            assert ws.recv() == '{"ack": 1}'            # この間に届いた ping には自動で pong を返している
            ws.close()
            th.join(3)
            ops = [op for op, _ in log]
            # テキストを送ってから ping を読むので、サーバーには TEXT → PONG → CLOSE の順に届く
            assert ops == [wc.OP_TEXT, wc.OP_PONG, wc.OP_CLOSE], ops
            assert log[1][1] == b"beat"
        except OSError as e:
            assert not ok and "403" in str(e), e
        srv.close()
    print("3. 小さなサーバーにつないで送受信（合言葉が違えば 403 で断られる）: OK")


if __name__ == "__main__":
    test_vectors()
    test_frames()
    test_live()
    print("すべて OK")

In [ ]:
%%writefile proto_analysis.py
#!/usr/bin/env python3
"""II-4：HTTP／MQTT（QoS0・QoS1）／WebSocket の遅延・欠損を比べる（実験A）

使い方: cd ~/home-lab/partII/ii4 && uv run python proto_analysis.py ii4_msgs.csv [--runs RUN ...] [--exclude RUN ...] [--deadline 5]
  ii4_msgs.csv は Pi の ii4_export.py で書き出したもの。--runs を省くと、方式を混ぜた実行（plan が2文字以上）を全部使う。
  欠損は plan の全長（全ブロック）で数える。途中で止めた実行は --exclude で外す。

やること
  1. 重複（同じ dev・run・seq）を数えて1つにし、plan から「送るはずだった通」を作って欠損を数える
  2. 時計合わせ：WebSocket の返事の4つの時刻から、Pico の時計と Pi の時計の差 θ(t) = a + b·t を推定する
     （往復の短い下位20%だけを使う。行きと帰りの差によるかたよりは平均しても消えないので、
      誤差の上限の目安は「使った交換のうち、いちばん長い往復（20%点）の半分」）
  3. 方式ごとの遅延の分位点（中央値・95%点・99%点）と平均、欠損率（Wilson とブロック・ブートストラップの区間）
  4. ブロック（60通）の中央値・95%点を単位にして「方式 ＋ 実行ごとの高さ ＋ 実行ごとの時間の傾き」で回帰し、
     6組の方式の差を Holm で補正する。Pico と Pi の時計の差は「実行ごとの高さと傾き」が吸収するので、方式の差は時計合わせに頼らない
  5. 統計の誤用の例：1通ずつを独立とみなす、平均で比べる、補正なしで多くの検定をする
"""
import argparse
import itertools
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
NAME = {"h": "HTTP", "m": "MQTT QoS0", "q": "MQTT QoS1", "w": "WebSocket"}
COL = {"h": "C0", "m": "C1", "q": "C2", "w": "C3"}                  # 方式ごとの色（どの図でも同じにする）
RNG = np.random.default_rng(0)


def load(path, runs, exclude):
    d = pd.read_csv(path)
    m = d[(d["via"] != "status") & d["seq"].notna()].copy()
    for c in ("seq", "blk", "k", "bn", "per"):
        m[c] = m[c].astype(int)
    if runs:
        m = m[m["run"].isin(runs)]
    else:
        m = m[m["plan"].str.len() > 1]                 # 方式を混ぜた実行だけ（障害の実験は1文字の plan）
    if exclude:
        m = m[~m["run"].isin(exclude)]
    if m.empty:
        sys.exit("対象の記録がありません（--runs を確かめる）")
    m = m.sort_values("recv_unix")
    m["dup"] = m.duplicated(["dev", "run", "seq"])     # 2回目以降に届いたもの（再送による重複）
    return m


def expected(m):
    """plan と bn から、送るはずだった通（ブロックごと）を作る。plan の全長で数える（最後のブロックが丸ごと欠けても数える）"""
    rows = []
    for (dev, run), g in m.groupby(["dev", "run"]):
        plan, bn = g["plan"].iloc[0], int(g["bn"].iloc[0])
        for b in range(len(plan)):
            for k in range(bn):
                rows.append((dev, run, b * bn + k, b, plan[b]))
    return pd.DataFrame(rows, columns=["dev", "run", "seq", "blk", "arm"])


def clock_fit(g):
    """θ = Pi の時計 − Pico の時計 [µs] を、往復の短い交換だけで a + b·(t − t0) に当てはめる"""
    s = g.dropna(subset=["x1", "x2", "x3", "x4"]).drop_duplicates("x1")
    x1, x2, x3, x4 = (s[c].astype(float).to_numpy() for c in ("x1", "x2", "x3", "x4"))
    theta = ((x2 - x1) + (x3 - x4)) / 2               # NTP と同じ式
    delta = (x4 - x1) - (x3 - x2)                     # 往復のうちネットワークにいた時間
    keep = delta <= np.quantile(delta, 0.2)           # 往復の短いもの（下位20%）だけ使う
    t0 = x1.min()
    b, a = np.polyfit(x1[keep] - t0, theta[keep], 1)
    resid = theta[keep] - (a + b * (x1[keep] - t0))
    return dict(a=a, b=b, t0=t0, n=len(s), n_use=int(keep.sum()), dmin=delta.min(), d20=delta[keep].max(),
                resid_sd=resid.std(ddof=2),
                x1=x1, theta=theta, delta=delta, keep=keep)


def block_boot_loss(e, arm, nb=2000):
    """ブロックを単位にしたブートストラップで、欠損率の 95% 区間（欠損がまとまって起きることを考える）"""
    g = e[e["arm"] == arm].groupby(["dev", "run", "blk"])["lost"].agg(["sum", "count"]).to_numpy()
    idx = RNG.integers(0, len(g), size=(nb, len(g)))
    r = g[idx, 0].sum(1) / g[idx, 1].sum(1)
    return np.quantile(r, [0.025, 0.975])


def perm_loss(blk, a, b, nperm=4000):
    """ブロックの欠損率の差の並べ替え検定（同じ実行の中で、ブロックの方式のラベルを入れ替える）"""
    x = blk[blk["arm"].isin([a, b])].copy()
    obs = x[x["arm"] == a]["loss"].mean() - x[x["arm"] == b]["loss"].mean()
    labels = [g["arm"].to_numpy() for _, g in x.groupby("devrun")]
    vals = [g["loss"].to_numpy() for _, g in x.groupby("devrun")]
    cnt = 0
    for _ in range(nperm):
        la = np.concatenate([RNG.permutation(l) for l in labels])
        v = np.concatenate(vals)
        diff = v[la == a].mean() - v[la == b].mean()
        cnt += abs(diff) >= abs(obs) - 1e-12
    return obs, (cnt + 1) / (nperm + 1)


def pair_tests(fit, arms, ref):
    """回帰の係数から、全部の組の差（a − b）とその p 値"""
    out = []
    names = list(fit.params.index)
    for a, b in itertools.combinations(arms, 2):
        L = np.zeros(len(names))
        for arm, sgn in ((a, 1), (b, -1)):
            if arm != ref:
                L[names.index(f"C(arm, Treatment('{ref}'))[T.{arm}]")] += sgn
        t = fit.t_test(L)
        lo, hi = np.asarray(t.conf_int()).ravel()
        out.append((a, b, float(np.asarray(t.effect).ravel()[0]), lo, hi, float(np.asarray(t.pvalue).ravel()[0])))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv")
    ap.add_argument("--runs", nargs="*", default=None)
    ap.add_argument("--exclude", nargs="*", default=None, help="途中で止めた実行など、外す run")
    ap.add_argument("--deadline", type=float, default=5.0, help="この秒数より遅れたら「間に合わなかった」と数える")
    a = ap.parse_args()
    m = load(a.csv, a.runs, a.exclude)
    first = m[~m["dup"]].copy()
    e = expected(m).merge(first[["dev", "run", "seq", "recv_unix"]], on=["dev", "run", "seq"], how="left")
    e = e[e["arm"] != "i"].copy()                     # i（何も送らないブロック）は欠損に数えない
    e["lost"] = e["recv_unix"].isna().astype(int)
    arms = [x for x in "hmqw" if x in set(e["arm"])]
    print(f"対象：{m.groupby(['dev', 'run']).ngroups} 回の実行（{', '.join(sorted(m['dev'].unique()))}）、"
          f"送るはずだった通 {len(e)}、届いた行 {len(m)}（うち重複 {int(m['dup'].sum())}）")
    if "ns" in m.columns:                              # 方式の切り替えで捨てた通（その方式の欠損に入っている）
        ns = int(m.groupby(["dev", "run"])["ns"].max().fillna(0).sum())
        print(f"  方式が切り替わったときに、前の方式で送れずに捨てた通：{ns}（それぞれの方式の欠損として数えている）")

    # ---- 2. 時計合わせ ----
    print("\n[時計合わせ] θ(t) = Pi − Pico の時計の差（WebSocket の返事から）")
    fits = {}
    for key, g in first.groupby(["dev", "run"]):
        f = clock_fit(g)
        fits[key] = f
        print(f"  {key[0]} {key[1]}: 交換 {f['n']} 回（使ったのは往復の短い {f['n_use']} 回）、"
              f"時計の進みの差 {f['b'] * 1e6:+.1f} ppm、当てはめの残差 SD {f['resid_sd'] / 1000:.2f} ms、"
              f"使った往復 {f['dmin'] / 1000:.1f}〜{f['d20'] / 1000:.1f} ms → 片道の誤差の上限の目安 ±{f['d20'] / 2000:.1f} ms")
    first["tg"] = first["tg"].astype(float)
    first["lat_ms"] = np.nan
    for key, idx in first.groupby(["dev", "run"]).groups.items():
        f = fits[key]
        tg = first.loc[idx, "tg"]
        theta = f["a"] + f["b"] * (tg - f["t0"])
        first.loc[idx, "lat_ms"] = (first.loc[idx, "recv_unix"] * 1e6 - (tg + theta)) / 1000
        # 時計合わせを使わない「相対の遅延」：受け取った時刻 − 予定の時刻（実行ごとに最小を 0 にする）
        raw = first.loc[idx, "recv_unix"] * 1000 - tg / 1000
        first.loc[idx, "rel_ms"] = raw - raw.min()

    # ---- 3. 方式ごとのまとめ ----
    print(f"\n[方式ごと] 遅延 [ms]（時計合わせで求めた片道。誤差は上の「上限」程度）、欠損率 [%]")
    print(f"{'方式':10s} {'届いた':>9s} {'欠損率':>6s} {'Wilson（独立とみなす）':>18s} {'ブロック単位':>14s} "
          f"{'中央値':>7s} {'95%点':>7s} {'99%点':>8s} {'平均':>7s} {'重複':>4s} {f'>{a.deadline:g}s':>5s}")
    summ = {}
    for arm in arms:
        ee = e[e["arm"] == arm]
        k, n = int(ee["lost"].sum()), len(ee)
        lo, hi = proportion_confint(k, n, method="wilson")
        blo, bhi = block_boot_loss(e, arm)
        x = first[first["arm"] == arm]["lat_ms"]
        dup = int(m[(m["arm"] == arm) & m["dup"]].shape[0])
        late = int((x > a.deadline * 1000).sum())
        summ[arm] = x
        print(f"{NAME[arm]:10s} {n - k:4d}/{n:<4d} {k / n * 100:6.2f} {f'{lo * 100:.2f}〜{hi * 100:.2f}':>18s} "
              f"{f'{blo * 100:.2f}〜{bhi * 100:.2f}':>14s} {x.median():7.1f} {x.quantile(.95):7.1f} "
              f"{x.quantile(.99):8.1f} {x.mean():7.1f} {dup:4d} {late:5d}")
    print("  欠損が0のとき、Wilson の上限は約 3.8/n、「3の法則」の上限は 3/n（どちらも独立を仮定）")

    # ---- Pico が測った送信の所要時間と、Pi での受信間隔 ----
    print("\n[Pico が自分の時計で測った、1通の送信にかかった時間 rtt（時計合わせ不要）と、Pi での受信間隔のゆらぎ]")
    r = first[(first["k"] > 0) & (first["try"] == 1)].dropna(subset=["rtt"])
    for arm in arms:
        x = r[r["arm"] == arm]["rtt"] / 1000
        what = {"h": "接続＋要求＋返事＋切断", "m": "書き込みだけ（往復ではない）", "q": "PUBACK までの往復",
                "w": "ack までの往復"}[arm]
        f2 = first[first["arm"] == arm].sort_values(["dev", "run", "seq"])
        dt = f2.groupby(["dev", "run", "blk"])["recv_unix"].diff()
        ok = f2.groupby(["dev", "run", "blk"])["seq"].diff() == 1
        jit = (dt[ok] * 1000 - f2["per"][ok])
        print(f"  {NAME[arm]:10s} rtt 中央値 {x.median():6.1f} ms、95%点 {x.quantile(.95):6.1f} ms（{what}）｜"
              f"受信間隔 − 送信間隔 の四分位範囲 {jit.quantile(.75) - jit.quantile(.25):5.1f} ms")

    # ---- 4. ブロックを単位にした比較 ----
    first["devrun"] = first["dev"] + "/" + first["run"]
    blk = (first.groupby(["devrun", "blk", "arm"])
           .agg(med=("rel_ms", "median"), p95=("rel_ms", lambda v: v.quantile(.95)), tg=("tg", "min"))
           .reset_index())
    blk["tmin"] = (blk["tg"] - blk.groupby("devrun")["tg"].transform("min")) / 60e6
    lossb = e.assign(devrun=e["dev"] + "/" + e["run"]).groupby(["devrun", "blk", "arm"])["lost"].mean()
    blk = blk.merge(lossb.rename("loss").reset_index(), on=["devrun", "blk", "arm"], how="right")
    ref = "m" if "m" in arms else arms[0]
    print(f"\n[ブロックの中央値・95%点の回帰]（{len(blk)} ブロック）: 値 ~ 方式 + 実行ごとの高さ + 実行ごとの時間の傾き、HC3 の標準誤差")
    tests = []
    for col, label in (("med", "中央値"), ("p95", "95%点")):
        fit = smf.ols(f"{col} ~ C(arm, Treatment('{ref}')) + C(devrun) + C(devrun):tmin",
                      blk.dropna(subset=[col])).fit(cov_type="HC3")
        for x1, x2, est, lo, hi, p in pair_tests(fit, arms, ref):
            tests.append((label, x1, x2, est, lo, hi, p, "ms"))
    for x1, x2 in itertools.combinations(arms, 2):
        obs, p = perm_loss(blk, x1, x2)
        tests.append(("欠損率", x1, x2, obs * 100, np.nan, np.nan, p, "%"))
    p = np.array([t[6] for t in tests])
    rej, padj, _, _ = multipletests(p, alpha=0.05, method="holm")
    for t, pa, rj in zip(tests, padj, rej):
        ci = "" if np.isnan(t[4]) else f"（95%区間 {t[4]:+.1f}〜{t[5]:+.1f}）"
        print(f"  {t[0]:4s} {NAME[t[1]]:>10s} − {NAME[t[2]]:<10s} = {t[3]:+7.2f} {t[7]}{ci}  "
              f"p = {t[6]:.4f}、Holm 後 {pa:.4f} {'＊' if rj else ''}")
    print(f"  【誤用の例】補正なしで p < 0.05 の数：{int((p < 0.05).sum())} / {len(p)}、Holm で補正すると {int(rej.sum())} / {len(p)}")
    print("  （欠損率は、同じ実行の中でブロックの方式のラベルを入れ替える並べ替え検定）")

    # ---- 5. 統計の誤用の例 ----
    print("\n[統計の誤用の例]")
    x1, x2 = ("h", "w") if {"h", "w"} <= set(arms) else (arms[0], arms[1])
    rr = first.sort_values(["devrun", "seq"])
    rho = rr.groupby(["devrun", "blk"])["rel_ms"].apply(lambda v: v.autocorr(1) if len(v) > 3 else np.nan).mean()
    n1 = int((rr["arm"] == x1).sum())
    print(f"  (1) 続けて送った遅延は独立ではない：ブロックの中のとなりどうしの相関の平均 ρ = {rho:.2f}"
          f" → {NAME[x1]} の {n1} 通は、独立な数にすると約 {n1 * (1 - rho) / (1 + rho):.0f} 通ぶん（AR(1) の近似）")
    pu = stats.mannwhitneyu(rr[rr["arm"] == x1]["rel_ms"], rr[rr["arm"] == x2]["rel_ms"]).pvalue
    pb = stats.mannwhitneyu(blk[blk["arm"] == x1]["med"].dropna(), blk[blk["arm"] == x2]["med"].dropna()).pvalue
    print(f"      Mann–Whitney：1通ずつを独立とみなすと p = {pu:.2g}、ブロックの中央値どうしなら p = {pb:.2g}")
    for arm in (x1, x2):
        x = summ[arm]
        cut = x[x <= x.quantile(0.99)]
        print(f"  (2) {NAME[arm]:10s} 平均 {x.mean():7.1f} ms、中央値 {x.median():6.1f} ms、上位1%を除いた平均 {cut.mean():6.1f} ms"
              f"（ごく一部の大きな遅延が平均を引っぱる。裾の重い分布は分位点で比べる）")
    pt = stats.ttest_ind(summ[x1], summ[x2], equal_var=False).pvalue
    print(f"      平均の t 検定（{NAME[x1]} と {NAME[x2]}、1通ずつ）p = {pt:.2g}：独立も正規も成り立たないので、この p は信用できない")

    # ---- 図 ----
    fig, ax = plt.subplots(2, 2, figsize=(12, 8.6))
    for arm in arms:
        x = np.sort(summ[arm].dropna().to_numpy())
        ax[0, 0].step(np.clip(x, 0.1, None), np.arange(1, len(x) + 1) / len(x), where="post", color=COL[arm], label=NAME[arm])
    ax[0, 0].set(xscale="log", xlabel="片道の遅延 [ms]（対数目盛。0.1 未満は 0.1 に）", ylabel="累積割合",
                 title="遅延の経験分布関数（右に長い裾）")
    for arm in arms:
        g = blk[blk["arm"] == arm]
        ax[0, 1].plot(g["tmin"], g["med"], "o", color=COL[arm], label=NAME[arm])
    ax[0, 1].set(xlabel="実行の始めからの時間 [分]", ylabel="ブロックの中央値（相対の遅延）[ms]",
                 title="ブロックの中央値（実行ごとの高さ・傾きは回帰で取り除く）")
    data = [r[r["arm"] == arm]["rtt"].to_numpy() / 1000 for arm in arms]
    ax[1, 0].boxplot(data, tick_labels=[NAME[x] for x in arms], whis=(5, 95), showfliers=False)
    ax[1, 0].set(yscale="log", ylabel="Pico が測った送信の所要時間 [ms]", title="1通の送信にかかった時間（5〜95%点）")
    f = next(iter(fits.values()))
    ax[1, 1].plot((f["x1"] - f["t0"]) / 60e6, f["theta"] / 1000 - f["a"] / 1000, ".", ms=3, alpha=0.4, label="すべての交換")
    ax[1, 1].plot((f["x1"][f["keep"]] - f["t0"]) / 60e6, f["theta"][f["keep"]] / 1000 - f["a"] / 1000, ".", ms=4,
                  label="往復の短い20%")
    tt = np.linspace(0, (f["x1"].max() - f["t0"]), 50)
    ax[1, 1].plot(tt / 60e6, f["b"] * tt / 1000, "-", lw=2, label=f"当てはめ（{f['b'] * 1e6:+.1f} ppm）")
    yk = f["theta"][f["keep"]] / 1000 - f["a"] / 1000
    ax[1, 1].set_ylim(yk.min() - 15, yk.max() + 15)      # 往復の長い交換（大きく外れる点）は枠の外
    ax[1, 1].set(xlabel="時間 [分]", ylabel="θ − a [ms]", title=f"時計の差 θ の推定（{next(iter(fits))[0]}）")
    for x in ax.ravel():
        x.grid(True, which="both", alpha=0.3)
    for x in (ax[0, 0], ax[0, 1], ax[1, 1]):
        x.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("proto.png", dpi=130)
    print("\nproto.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile current_analysis.py
#!/usr/bin/env python3
"""II-4：方式ごとの Pico B の消費電流を比べる（実験B）

使い方: cd ~/home-lab/partII/ii4 && uv run python current_analysis.py data/meter_XXXX.csv ii4_msgs.csv [--run RUN] [--guard 5]
  data/meter_XXXX.csv … Pico A（ina_meter.py）の記録を serial_logger.py で保存したもの（host_time は Mac の時刻）
  ii4_msgs.csv       … Pi の ii4_export.py で書き出したもの。Pico B（pico2-ii4）の、plan に i を含む実行を使う
ブロックの時間は、Pico B の予定の時刻 tg を「時計の差 θ(t)」で Pi の時刻に直して求める（i のブロックは何も届かないため）。
Mac と Pi の時計はどちらも NTP で合わせてあり、差は 1秒よりずっと小さいとみなし、ブロックの前後 --guard 秒を捨てる。
"""
import argparse
import itertools
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
NAME = {"h": "HTTP", "m": "MQTT QoS0", "q": "MQTT QoS1", "w": "WebSocket", "i": "送らない"}
COL = {"h": "C0", "m": "C1", "q": "C2", "w": "C3", "i": "C7"}      # 方式ごとの色（proto_analysis.py と同じ）


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("meter")
    ap.add_argument("msgs")
    ap.add_argument("--dev", default="pico2-ii4")
    ap.add_argument("--run", default=None)
    ap.add_argument("--guard", type=float, default=5.0)
    a = ap.parse_args()
    mt = pd.read_csv(a.meter)
    d = pd.read_csv(a.msgs)
    d = d[(d["via"] != "status") & (d["dev"] == a.dev) & d["seq"].notna()]
    if a.run is None:
        cand = d[d["plan"].str.contains("i", na=False)]
        if cand.empty:
            sys.exit("plan に i を含む実行がありません（node.py の ARMS を \"hmqwi\" にして実験B を行う）")
        a.run = cand.sort_values("recv_unix")["run"].iloc[-1]
    g = d[d["run"] == a.run].drop_duplicates("seq")
    plan, bn, per = g["plan"].iloc[0], int(g["bn"].iloc[0]), int(g["per"].iloc[0])
    # 時計の差 θ（proto_analysis.py と同じ方法）
    s = g.dropna(subset=["x1", "x2", "x3", "x4"])
    x1, x2, x3, x4 = (s[c].astype(float).to_numpy() for c in ("x1", "x2", "x3", "x4"))
    theta, delta = ((x2 - x1) + (x3 - x4)) / 2, (x4 - x1) - (x3 - x2)
    keep = delta <= np.quantile(delta, 0.2)
    b, a0 = np.polyfit(x1[keep], theta[keep], 1)
    tg0 = float(g["tg"].astype(float).min() - g["seq"].min() * per * 1000)     # seq = 0 の予定の時刻
    to_pi = lambda tg: (tg + a0 + b * tg) / 1e6
    print(f"実行 {a.run}：plan = {plan}、{len(plan)} ブロック × {bn} 通、時計合わせに {int(keep.sum())} 回の交換")

    rows = []
    for k, arm in enumerate(plan):
        t_s = to_pi(tg0 + k * bn * per * 1000) + a.guard
        t_e = to_pi(tg0 + (k + 1) * bn * per * 1000) - a.guard
        w = mt[(mt["host_time"] >= t_s) & (mt["host_time"] < t_e)]
        if len(w) < 10:
            continue
        rows.append(dict(blk=k, arm=arm, tmin=(t_s - to_pi(tg0)) / 60, i=w["i_mean_ma"].mean(), n=len(w),
                         bus_v=w["bus_v"].mean()))
    blk = pd.DataFrame(rows)
    if blk.empty:
        sys.exit("ブロックの時間に電流の記録が重なりません（Mac と Pi の時計、記録の時間を確かめる）")
    print(f"\n[方式ごとのブロックの平均電流]（各ブロックの前後 {a.guard:g} 秒を捨てた）")
    for arm, x in blk.groupby("arm")["i"]:
        print(f"  {NAME[arm]:10s} {len(x)} ブロック  平均 {x.mean():6.2f} mA（ブロックどうしの SD {x.std():.2f}）")

    ref = "i" if "i" in plan else blk["arm"].iloc[0]
    fit = smf.ols(f"i ~ C(arm, Treatment('{ref}')) + tmin", blk).fit(cov_type="HC3")
    print(f"\n[回帰] 電流 ~ 方式 + 時間（温度などによるゆっくりした変化）：時間の係数 {fit.params['tmin']:+.3f} mA/分")
    names = list(fit.params.index)
    arms = [x for x in "ihmqw" if x in set(blk["arm"])]
    res = []
    for x, y in itertools.combinations(arms, 2):
        L = np.zeros(len(names))
        for arm, sg in ((x, 1), (y, -1)):
            if arm != ref:
                L[names.index(f"C(arm, Treatment('{ref}'))[T.{arm}]")] += sg
        t = fit.t_test(L)
        lo, hi = np.asarray(t.conf_int()).ravel()
        res.append((y, x, -float(np.asarray(t.effect).ravel()[0]), -hi, -lo, float(np.asarray(t.pvalue).ravel()[0])))
    rej, padj, _, _ = multipletests([r[5] for r in res], method="holm")
    vbus = blk["bus_v"].mean()
    for (y, x, est, lo, hi, p), pa, rj in zip(res, padj, rej):
        extra = ""
        if x == "i":                                   # 送らないときとの差 → 1通あたりのエネルギー
            extra = f" → 1通あたり {est * vbus * per / 1000:.1f} mJ（{lo * vbus * per / 1000:.1f}〜{hi * vbus * per / 1000:.1f}）"
        print(f"  {NAME[y]:>10s} − {NAME[x]:<10s} = {est:+6.2f} mA（95%区間 {lo:+.2f}〜{hi:+.2f}）"
              f" Holm 後 p = {pa:.4f} {'＊' if rj else ''}{extra}")
    print(f"  1通あたりのエネルギー = 電流の差 × バス電圧（{vbus:.2f} V）× 送る間隔（{per / 1000:g} 秒）")
    print("  注意：INA219 の値は約0.5ms ごとの平均。短いピークは平均に入るが、形は見えない（II-4-3-6）")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.3), gridspec_kw={"width_ratios": [2, 1]})
    t0 = to_pi(tg0)
    ax[0].plot((mt["host_time"] - t0) / 60, mt["i_mean_ma"], lw=0.5, color="0.5")
    for arm in arms:
        x = blk[blk["arm"] == arm]
        ax[0].plot(x["tmin"] + (bn * per / 1000 - 2 * a.guard) / 120, x["i"], "o", color=COL[arm], label=NAME[arm])
    ax[0].set(xlim=(-1, len(plan) * bn * per / 60000 + 1), xlabel="実行の始めからの時間 [分]",
              ylabel="Pico B の電流 [mA]", title="100ms ごとの平均（灰）とブロックの平均（点）")
    ax[1].boxplot([blk[blk["arm"] == x]["i"] for x in arms], tick_labels=[NAME[x] for x in arms])
    ax[1].set(ylabel="ブロックの平均電流 [mA]", title="方式ごとのブロックの平均")
    ax[1].tick_params(axis="x", rotation=25)
    for x in ax:
        x.grid(True, alpha=0.3)
    ax[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("current.png", dpi=130)
    print("\ncurrent.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile fault_timeline.py
#!/usr/bin/env python3
"""II-4：障害を起こした実験（中継機を抜く・受け口を止める・遠ざける）の時間の流れを見る

使い方: cd ~/home-lab/partII/ii4 && uv run python fault_timeline.py ii4_msgs.csv marks.csv [--runs RUN ...] [--deadline 5]
  marks.csv … mac_ii4.sh の mark で記録した「Mac の時刻,ラベル」。ラベルは down:名前 と up:名前 を組にする
  --runs を省くと、方式を1つに固定した実行（plan が1文字）を全部使う。
重複・遅れ（>deadline）・最大の遅れは、障害ごとに「止めた時刻〜戻して AFTER_S 秒後」に作った通だけで数える。
時計合わせ：MQTT だけの実行には返事の時刻がないので、障害の前（最初の 120 秒）でいちばん早く届いた通の遅れを 0 とみなし、
  「最小からの遅れ（相対の遅延）」で見る（秒の単位の現象を見るには十分。ms の比較には proto_analysis.py を使う）。
"""
import argparse
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
NAME = {"h": "HTTP", "m": "MQTT QoS0", "q": "MQTT QoS1", "w": "WebSocket"}
AFTER_S = 60      # 戻してからこの秒数までに作った通も、その障害の影響として数える


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv")
    ap.add_argument("marks")
    ap.add_argument("--runs", nargs="*", default=None)
    ap.add_argument("--deadline", type=float, default=5.0)
    a = ap.parse_args()
    d = pd.read_csv(a.csv)
    st = d[d["via"] == "status"]
    m = d[(d["via"] != "status") & d["seq"].notna()].copy()
    m = m[m["run"].isin(a.runs)] if a.runs else m[m["plan"].str.len() == 1]
    if m.empty:
        sys.exit("対象の実行がありません")
    mk = pd.read_csv(a.marks, header=None, names=["t", "label"])
    pairs = []
    for _, r in mk[mk["label"].str.startswith("down:")].iterrows():
        name = r["label"][5:]
        up = mk[(mk["label"] == "up:" + name) & (mk["t"] > r["t"])]
        if len(up):
            pairs.append((name, r["t"], up["t"].iloc[0]))
    runs = list(m.groupby(["dev", "run"]))
    fig, axs = plt.subplots(len(runs), 1, figsize=(11, 2.6 * len(runs)), squeeze=False)
    print(f"{'方式':10s} {'障害':10s} {'止めた[s]':>8s} {'その間の通':>9s} {'届いた':>6s} {'欠損':>4s} {'重複':>4s} "
          f"{f'>{a.deadline:g}s':>5s} {'最大の遅れ[s]':>12s} {'戻してから最初に届くまで[s]':>24s} {'たまった分を送りきるまで[s]':>24s}")
    for ax, ((dev, run), g) in zip(axs[:, 0], runs):
        g = g.sort_values("recv_unix")
        dupseq = g.loc[g.duplicated("seq"), "seq"]     # 2回目以降に届いた通の seq（重複）
        g = g.drop_duplicates("seq").copy()
        arm, bn, per = g["arm"].iloc[0], int(g["bn"].iloc[0]), int(g["per"].iloc[0])
        raw = g["recv_unix"] - g["tg"].astype(float) / 1e6
        early = g["recv_unix"] < g["recv_unix"].min() + 120
        off = raw[early].min()                         # 最初の 120 秒でいちばん早く届いた通の遅れを 0 とみなす
        g["t_gen"] = g["tg"].astype(float) / 1e6 + off  # 作った時刻（Pi の時計に直したもの）
        g["late"] = g["recv_unix"] - g["t_gen"]
        tg0 = g["t_gen"].iloc[0] - g["seq"].iloc[0] * per / 1000
        allseq = pd.DataFrame({"seq": range(bn)})
        allseq["t_gen"] = tg0 + allseq["seq"] * per / 1000
        lost = allseq[~allseq["seq"].isin(g["seq"])]
        t_ref = None
        for name, t_down, t_up in pairs:
            if not (allseq["t_gen"].min() <= t_down <= allseq["t_gen"].max()):
                continue
            t_ref = t_ref or t_down
            inwin = allseq[(allseq["t_gen"] >= t_down) & (allseq["t_gen"] < t_up)]
            got = g[g["seq"].isin(inwin["seq"])]
            # 重複・遅れ・最大の遅れは、障害の間と、戻してから AFTER_S 秒までに作った通に限って数える
            win = g[(g["t_gen"] >= t_down) & (g["t_gen"] < t_up + AFTER_S)]
            dup = int(dupseq.isin(win["seq"]).sum())
            after = g[g["recv_unix"] > t_up]
            first = after["recv_unix"].min() - t_up if len(after) else np.nan
            backlog = g[g["t_gen"] < t_up]["recv_unix"].max() - t_up
            print(f"{NAME[arm]:10s} {name:10s} {t_up - t_down:8.1f} {len(inwin):9d} {len(got):6d} "
                  f"{len(inwin) - len(got):4d} {dup:4d} {int((win['late'] > a.deadline).sum()):5d} {win['late'].max():12.1f} "
                  f"{first:24.1f} {backlog:24.1f}")
            ax.axvspan(0 if t_ref == t_down else t_down - t_ref, t_up - t_ref, color="0.85")
        t_ref = t_ref or g["t_gen"].iloc[0]
        ax.semilogy(g["t_gen"] - t_ref, g["late"].clip(lower=1e-3), ".", ms=3, label="届いた通（最小からの遅れ）")
        ax.plot(lost["t_gen"] - t_ref, np.full(len(lost), 2e-3), "|", color="C3", ms=12, label="届かなかった通")
        lw = st[(st["dev"] == dev) & (st["run"] == run) & (st["state"] == "lwt")]
        for t in lw["recv_unix"]:
            last = g[g["recv_unix"] < t]["recv_unix"].max()
            ax.axvline(t - t_ref, color="C2", ls="--", label=f"LWT（最後に届いてから {t - last:.1f} 秒）")
            print(f"  {NAME[arm]}：LWT が出たのは、最後の通が届いてから {t - last:.1f} 秒後（keepalive の 1.5 倍が目安）")
        ax.set(ylabel="遅れ [s]", title=f"{NAME[arm]}（{dev} {run}）。灰色 = 障害を起こしていた時間", ylim=(1e-3, None))
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=7, loc="upper right")
    axs[-1, 0].set_xlabel("障害を起こした時刻からの時間 [s]（作った時刻で並べる）")
    fig.tight_layout()
    fig.savefig("fault.png", dpi=130)
    print("fault.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile synth_ii4.py
#!/usr/bin/env python3
"""II-4 ノートブック版で追加：Pico・Pi がなくても解析を試すための合成の記録（本文にはありません）

作るもの（教科書の ii4_export.py・mark・serial_logger.py と同じ形）
  ii4_msgs.csv       … 実験A（Pico 2台 × 2回、4方式 × 4ブロック × 60通、順番は乱数）、
                       実験C（方式を1つに固定した 400 通の実行 × 4方式。180〜240 秒に中継機を抜いた）、
                       実験B（Pico B、ARMS = "hmqwi" の 5種類 × 4ブロック）の届いた行と、LWT の状態の行
  marks.csv          … 実験C の down:repeater・up:repeater の印（Mac の時刻）
  data/meter_syn.csv … 実験B の INA219 の 100ms ごとの平均電流（Pico A の ina_meter.py の代わり）
しくみ（どれも仮の値）
  遅延 = 方式ごとの基準（HTTP 18ms、QoS0 6ms、QoS1 6.5ms、WebSocket 8ms）× exp(AR(1) のゆらぎ、ρ = 0.6)
         ＋ パレート分布の裾 ＋ 平均 120 秒に1回の「しゃっくり」（1〜4秒、0.2〜1.5秒の詰まり。QoS0 は半分が欠ける）
  Pico の時計は Pi の時計より 25 ppm 速い。WebSocket の ack から 4つの時刻（x1〜x4）を作る
  中継機を抜いた間：QoS0 は最初の 20 秒の通が欠け、ほかはバッファから送り直す（再送で重複も少し出る）
  電流 = 46 mA ＋ 方式ごとの増分（HTTP 3.0、QoS0 1.0、QoS1 1.3、WebSocket 1.1 mA）＋ ゆっくりした変化 ＋ AR(1) の雑音 ＋ まれなパルス
使い方: python synth_ii4.py
"""
import csv, json, math, os, random
import numpy as np
rng = np.random.default_rng(42); R = random.Random(42)
BASE = {"h": 0.018, "m": 0.006, "q": 0.0065, "w": 0.008}
EXTRA_I = {"h": 3.0, "m": 1.0, "q": 1.3, "w": 1.1, "i": 0.0}
rows, status, marks, meter = [], [], [], []
def plan_of(arms, nb):
    p = list(arms * nb); R.shuffle(p); return "".join(p)
def run_one(dev, t_start, arms="hmqw", nb=4, bn=60, fixed=None, outage=None, ppm=25e-6):
    run = "%08x" % R.getrandbits(32)
    plan = fixed[0] if fixed else plan_of(arms, nb)
    if fixed: bn = fixed[1]
    B0 = t_start - 20.0                      # 起動の時刻（サーバー時計）
    tg0 = int(20.0 * (1 + ppm) * 1e6)
    n = len(plan) * bn
    # AR(1) の対数ゆらぎ
    z = np.zeros(n); e = rng.normal(0, 0.35, n)
    for i in range(1, n): z[i] = 0.6 * z[i-1] + e[i]
    # しゃっくり（Wi-Fi の一時的な詰まり）
    hic = []
    t = t_start
    while t < t_start + n:
        t += rng.exponential(120); hic.append((t, t + rng.uniform(1, 4), rng.uniform(0.2, 1.5)))
    last_sync = None; last_rtt = None; last_recv = None
    pending = []
    for s in range(n):
        blk, k = divmod(s, bn); arm = plan[blk]
        tg = tg0 + s * 1_000_000
        gen = B0 + tg / 1e6 / (1 + ppm)
        lat = BASE.get(arm, 0.01) * math.exp(z[s]) + (rng.pareto(2.2) * 0.003)
        lost = False
        for (a, b, extra) in hic:
            if a <= gen < b:
                if arm == "m" and rng.random() < 0.5: lost = True
                lat += extra * rng.uniform(0.3, 1)
        if arm == "m" and rng.random() < 0.002: lost = True
        if outage and outage[0] <= gen < outage[1] + 6:
            if arm == "m" and gen < outage[0] + 20: lost = True
            else: lat = max(lat, outage[1] + 6 + rng.uniform(0, 8) - gen)
        if arm == "i": continue
        recv = gen + lat
        d = {"dev": dev, "run": run, "plan": plan, "bn": bn, "per": 1000, "seq": s, "blk": blk, "k": k, "arm": arm,
             "tg": tg, "ts": tg + 300, "try": 1 if lat < 1 else 2, "rssi": -58, "rtt": last_rtt, "nf": 0, "nc": 1, "nd": 0,
             "buf": 0, "temp_c": 22.1, "press_pa": 100800.0, "hum_pct": 45.0}
        if last_sync: d.update(dict(zip(("x1", "x4", "x2", "x3"), last_sync)))
        up = lat; down_ = BASE.get(arm, .01) * math.exp(rng.normal(0, .3)) + 0.0005
        rt = up + down_ + (0.010 if arm == "h" else 0)
        last_rtt = int(rt * 1e6) if arm != "m" else int(rng.uniform(300, 900))
        if arm == "w" and not lost:
            x1 = tg + 300; x2 = int((gen + 0.0003 * 0 + up) * 1e6); x3 = x2 + 50
            x4 = x1 + int(rt * (1 + ppm) * 1e6)
            last_sync = (x1, x4, x2, x3)
        if lost: continue
        via = {"h": "http", "w": "ws", "m": "mqtt", "q": "mqtt"}[arm]
        topic = {"http": f"http/{dev}", "ws": f"ws/{dev}", "mqtt": f"lab/{dev}/telemetry"}[via]
        rows.append({"recv_unix": f"{recv:.6f}", "via": via, "topic": topic, **d})
        if arm in "hq" and lat > 0.5 and rng.random() < 0.3:
            rows.append({"recv_unix": f"{recv + 0.4:.6f}", "via": via, "topic": topic, **{**d, "try": 3}})
        if outage and arm in "mq" and gen < outage[0]: last_recv = recv
    if outage and arm in "mq":
        status.append({"recv_unix": f"{last_recv + 15.2:.6f}", "via": "status", "topic": f"lab/{dev}/status",
                       "dev": dev, "run": run, "state": "lwt"})
    return run, B0, tg0, plan, bn
T = 1.7953e9
r1 = run_one("pico1-ii4", T); r2 = run_one("pico2-ii4", T + 2)
r3 = run_one("pico1-ii4", T + 3600); r4 = run_one("pico2-ii4", T + 3605)
t = T + 7200
for arm in "hqwm":
    out = (t + 180, t + 240)
    run_one("pico2-ii4", t, fixed=(arm, 400), outage=out)
    marks.append((out[0] + 0.05, "down:repeater")); marks.append((out[1] + 0.05, "up:repeater"))
    t += 600
run, B0, tg0, plan, bn = run_one("pico2-ii4", t, arms="hmqwi", nb=4)
start = B0 + tg0 / 1e6 / (1 + 25e-6)
ar = 0.0
for w in range(int(len(plan) * bn * 10) + 200):
    ht = start - 10 + w * 0.1
    blk = int((ht - start) // bn); arm = plan[blk] if 0 <= blk < len(plan) else "i"
    ar = 0.9 * ar + rng.normal(0, 0.3)
    i = 46.0 + EXTRA_I[arm] + 0.3 * (ht - start) / 1200 + ar + (rng.random() < 0.05) * rng.uniform(5, 30)
    meter.append((f"{ht:.3f}", int(w * 100), f"{i:.3f}", f"{i - 2:.2f}", f"{i + 25:.2f}", 180, "4.872"))
cols = ["recv_unix", "via", "topic", "dev", "run", "seq", "blk", "k", "arm", "tg", "ts", "try"]
allk = []
for r in rows + status:
    for k in r:
        if k not in allk: allk.append(k)
cols += sorted(k for k in allk if k not in cols)
rows.sort(key=lambda r: float(r["recv_unix"]))
with open("ii4_msgs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(sorted(rows + status, key=lambda r: float(r["recv_unix"])))
with open("marks.csv", "w") as f:
    for t, l in marks: f.write(f"{t:.3f},{l}\n")
os.makedirs("data", exist_ok=True)
with open("data/meter_syn.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["host_time", "t_ms", "i_mean_ma", "i_min_ma", "i_max_ma", "n", "bus_v"]); w.writerows(meter)
print(len(rows), "rows;", len(status), "status; current run", run, plan)

In [ ]:
import glob, os, re, shlex, subprocess, sys
from IPython.display import Image, display


def run(args):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    words = []
    for w in shlex.split(args):
        words += (sorted(glob.glob(w)) or [w]) if any(c in w for c in "*?[") else [w]
    p = subprocess.run([sys.executable] + words, capture_output=True, text=True)
    print(p.stdout, end="")
    err = "\n".join(l for l in p.stderr.splitlines() if "findfont" not in l and "font cache" not in l)
    if err.strip():
        print(err, file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def show(*paths, width=860):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=f, width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))


print("準備ができました")

## 1. WebSocket クライアントを検算する（手順 1）

`test_wsclient.py` は、Pico も Pi も使わずに、`pico/wsclient.py` を CPython で確かめます。(1) RFC 6455 の例（鍵 `dGhlIHNhbXBsZSBub25jZQ==` → `s3pPLMBiTxaQ9kYGzzhZRbK+xOo=`、マスク付きの "Hello"、長さの書き方）、(2) サーバー役のフレームを流し込んだときの分割・ping→pong・close・決まり違反（1002）、(3) 同じ計算機の中の小さなサーバー（別スレッド）につないで送受信し、合言葉が違えば 403 で断られること。

確かめること：「1. RFC 6455 の例と一致: OK」から「すべて OK」までの4行が出る。

In [ ]:
out = run("test_wsclient.py")
lines = [l for l in out.splitlines() if l.strip()]
ok(lines[0].startswith("1. RFC 6455 の例と一致: OK") and lines[-1] == "すべて OK" and len(lines) == 4,
   "「1. RFC 6455 の例と一致: OK」から「すべて OK」までの4行が出た")

## 2. ループバックで HTTP と WebSocket の往復を比べる（ノートブック版で追加）

同じ計算機の中（127.0.0.1）に、HTTP の受け口（`POST /ingest` の代わり。`Connection: close` で1通ごとに切る）と、WebSocket の受け口（`/ws` の代わり。届いたテキストに `{"ack": seq, "t2": …, "t3": …}` を返す）を別スレッドで立て、約380バイトの JSON を 1通ずつ送って往復の時間を測ります。方式の順番は、本文と同じく 50 通のブロックを乱数で混ぜます。

- HTTP は1通ごとに TCP の接続（3回のあいさつ）と切断、数百バイトの見出しが付きます。WebSocket は最初に1回だけ Upgrade し、あとは見出し 8 バイトのフレームだけです（II-4-3-1）。
- ここには Wi-Fi も Pi もないので、値は「接続を作るコストと見出しの処理」だけです。本文の結果の例（HTTP の rtt は数十 ms）よりずっと小さく、計算機で大きく変わります。

In [ ]:
import http.client, http.server, json, socket, threading, time
import numpy as np
sys.path.insert(0, "pico")
import wsclient as wc
from test_wsclient import read_frame            # サーバーの立場でクライアントのフレームを読む（test_wsclient.py の関数）

REQ_HDR = []                                     # HTTP の要求の見出しのバイト数


class Ingest(http.server.BaseHTTPRequestHandler):
    """api.py の POST /ingest の代わり（DB には書かない）"""
    protocol_version = "HTTP/1.1"

    def do_POST(self):
        t2 = time.time_ns() // 1000
        REQ_HDR.append(len(self.requestline) + 2 + sum(len(k) + len(v) + 4 for k, v in self.headers.items()) + 2)
        d = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        r = json.dumps({"ok": True, "seq": d["seq"], "t2": t2, "t3": time.time_ns() // 1000}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(r)))
        self.send_header("Connection", "close")
        self.end_headers()
        self.wfile.write(r)

    def log_message(self, *a):
        pass


def ws_server(srv):
    """api.py の /ws の代わり：ハンドシェイク → 届いたテキストに ack を返す → close で終わる"""
    conn, _ = srv.accept()
    conn.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
    req = b""
    while not req.endswith(b"\r\n\r\n"):
        req += conn.recv(1)
    h = {k.strip().lower(): v.strip() for k, v in (l.split(":", 1) for l in req.decode().split("\r\n")[1:] if ":" in l)}
    acc = wc.accept_key(h["sec-websocket-key"].encode()).decode()
    conn.sendall(("HTTP/1.1 101 Switching Protocols\r\nUpgrade: websocket\r\nConnection: Upgrade\r\n"
                  "Sec-WebSocket-Accept: %s\r\n\r\n" % acc).encode())
    while True:
        fin, op, data = read_frame(conn)
        if op == wc.OP_CLOSE:
            conn.sendall(bytes([0x88, 0x02]) + data[:2])
            break
        t2 = time.time_ns() // 1000
        seq = json.loads(data)["seq"]
        r = json.dumps({"ack": seq, "t2": t2, "t3": time.time_ns() // 1000}).encode()
        conn.sendall(bytes([0x81, len(r)]) + r)          # サーバーからのフレームはマスクしない（125 バイト以下）
    conn.close()


hs = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Ingest)
threading.Thread(target=hs.serve_forever, daemon=True).start()
ws_srv = socket.socket()
ws_srv.bind(("127.0.0.1", 0))
ws_srv.listen(1)
threading.Thread(target=ws_server, args=(ws_srv,), daemon=True).start()
ws = wc.connect("127.0.0.1", ws_srv.getsockname()[1], "/ws?dev=nb", timeout=3)


def body(seq):
    """約380バイトの JSON（本文の通と同じくらいの大きさ）"""
    d = {"dev": "pico-nb", "run": "0badc0de", "seq": seq, "arm": "x", "tg": 20000000 + seq * 1000000,
         "temp_c": 22.1, "press_pa": 100800.0, "hum_pct": 45.0, "pad": ""}
    d["pad"] = "x" * (380 - len(json.dumps(d)))
    return json.dumps(d)


def send_http(seq):
    t0 = time.perf_counter()
    c = http.client.HTTPConnection("127.0.0.1", hs.server_address[1], timeout=3)   # 1通ごとに接続（requests と同じ）
    c.request("POST", "/ingest", body=body(seq), headers={"Content-Type": "application/json", "Connection": "close"})
    r = c.getresponse()
    assert r.status == 200 and json.loads(r.read())["seq"] == seq
    c.close()
    return time.perf_counter() - t0


def send_ws(seq):
    t0 = time.perf_counter()
    ws.send_text(body(seq))
    assert json.loads(ws.recv())["ack"] == seq
    return time.perf_counter() - t0


N_BLOCK, BN = 8, 50                              # 2方式 × 4ブロック × 50通（本文の実験 A は 4方式 × 4ブロック × 60通）
rng = np.random.default_rng(1)
plan = "".join(rng.permutation(list("hw" * (N_BLOCK // 2))))
rtt = {"h": [], "w": []}
seq = 0
for arm in plan:
    for k in range(BN):
        rtt[arm].append((send_http if arm == "h" else send_ws)(seq) * 1000)
        seq += 1
ws.close()
hs.shutdown()
print("plan =", plan)
for arm, name in (("h", "HTTP（1通ごとに接続）"), ("w", "WebSocket（つないだまま）")):
    x = np.array(rtt[arm])
    print(f"{name:22s} {len(x)} 通  往復の中央値 {np.median(x):.3f} ms、95%点 {np.quantile(x, .95):.3f} ms")
ratio = np.median(rtt["h"]) / np.median(rtt["w"])
print(f"中央値の比 HTTP / WebSocket = {ratio:.1f} 倍")
ok(len(rtt["h"]) == len(rtt["w"]) == N_BLOCK // 2 * BN, "両方の方式で全部の通に返事が来た（ack の seq も一致）")
ok(np.median(rtt["h"]) > np.median(rtt["w"]), "1通ごとに接続する HTTP のほうが、往復の中央値が長い")
p = body(0).encode()
ok(len(p) == 380 and len(wc.build_frame(wc.OP_TEXT, p)) - len(p) == 8,
   "約380バイトの JSON なら、WebSocket のフレームの見出しは 8 バイト（2 ＋ 拡張長 2 ＋ マスクの鍵 4）")
ok(min(REQ_HDR) > 100, f"HTTP の要求の見出しは毎回 {min(REQ_HDR)} バイト（返事の見出しも別に付く）")

## 3. 原理の数値を確かめる（II-4-3）

本文に出てくる数値を、式どおりに計算し直します。バックオフは `node.py` と同じ「待ち時間の半分は決まった値、半分は乱数」（`backoff / 2 + backoff / 2 × 一様乱数`）で、乱数のシミュレーションでも確かめます。

In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportion_confint
wrap = 2**30 / 1e6
ok(abs(wrap - 1073.7) < 0.1 and abs(wrap / 60 - 17.9) < 0.05, f"ticks_us は {wrap:.1f} 秒（約 {wrap / 60:.1f} 分）で一周する。半周は約 {wrap / 120:.1f} 分")
ok(abs(960 * 25e-6 - 0.024) < 1e-12, "25 ppm の時計のずれは、16分（960秒）で 24 ms")
steps = [0.5, 1, 2, 4, 8, 16, 30]
mean_total = 0.75 * sum(steps)
sim = sum(b / 2 + b / 2 * np.random.default_rng(0).random(100_000) for b in steps)
ok(abs(mean_total - 46) < 0.5 and abs(sim.mean() - mean_total) < 0.1,
   f"7回試すまでの待ち時間の合計：式 0.75 × {sum(steps)} = {mean_total:.1f} 秒、シミュレーションの平均 {sim.mean():.1f} 秒（約 46 秒）")
neff = 1200 * (1 - 0.3) / (1 + 0.3)
ok(round(neff) == 646, f"ρ = 0.3、n = 1200 の有効サンプル数 n(1−ρ)/(1+ρ) ≈ {neff:.0f}")
lo, hi = proportion_confint(0, 1200, method="wilson")
ok(abs(hi * 100 - 0.32) < 0.005 and abs(3.84 / 1203.84 * 100 - 0.32) < 0.005,
   f"欠損 0 / 1200 の Wilson の上限 {hi * 100:.3f} %（約 3.84/1203.84 = 0.32 %、3の法則なら {3 / 1200 * 100:.2f} %）")
fw = 1 - 0.95**18
ok(abs(fw - 0.60) < 0.01, f"18回の検定で少なくとも1つが偶然 p < 0.05 になる確率 1 − 0.95^18 = {fw:.2f}")
ok(abs(0.05 / 18 - 0.0028) < 0.0001, f"Holm のいちばん厳しい基準 0.05/18 = {0.05 / 18:.4f}")
ok(abs(3e-3 * 4.87 * 1 * 1000 - 14.6) < 0.05, "電流の差 3 mA × 4.87 V × 1 秒 ≈ 14.6 mJ（約 15 mJ）")

## 4. 合成の記録を作る（実験 A・B・C の代わり）

`synth_ii4.py` が、`ii4_msgs.csv`（届いた行と LWT の行）、`marks.csv`（中継機を抜いた印）、`data/meter_syn.csv`（100ms ごとの電流）を作ります（数秒）。中身の作り方はファイルの先頭の説明のとおりです。

| 実行 | 中身 | 使う節 |
|---|---|---|
| 実験 A | pico1-ii4・pico2-ii4 × 2回。4方式 × 4ブロック × 60通（plan は乱数） | 5 |
| 実験 B | pico2-ii4。ARMS = "hmqwi"（送らないブロック i を含む）× 4ブロック | 5・6 |
| 実験 C | pico2-ii4。方式を1つに固定した 400 通の実行 × 4方式。始めから 180〜240 秒に中継機を抜いた | 7 |

`proto_analysis.py` は「方式を混ぜた実行（plan が2文字以上）」を全部使うので、5 節では実験 B の実行も（i のブロックを除いて）入ります。

In [ ]:
out = run("synth_ii4.py")
import pandas as pd
d = pd.read_csv("ii4_msgs.csv")
m = d[d["via"] != "status"]
print(d.groupby(["dev", "run"])["plan"].first())
ok(set(d["via"]) == {"http", "mqtt", "ws", "status"}, "via は http・mqtt・ws と、LWT の status")
ok((m.groupby(["dev", "run"])["plan"].first().str.len() > 1).sum() == 5, "方式を混ぜた実行が 5 回（実験 A の 4 回 ＋ 実験 B の 1 回）")
ok((m.groupby(["dev", "run"])["plan"].first().str.len() == 1).sum() == 4, "方式を1つに固定した実行が 4 回（実験 C）")

## 5. 遅延と欠損を比べる（手順 12、`proto_analysis.py`）

確かめること（手順 12）：「時計合わせの行に『時計の進みの差 ±数十 ppm 以内』『片道の誤差の上限の目安 ±数ms』。方式ごとの表、18個の比較と Holm の補正、誤用の例3つが表示され、proto.png ができる」。

本文 II-4-7 の結果の例（合成データ）：「中央値が MQTT・WebSocket より十ms 前後大きくなりやすい（合成データでは HTTP 約21ms、ほかは約9〜11ms）」。合成の記録では Pico の時計を 25 ppm 速くしてあるので、θ(t) の傾き（Pi − Pico）は約 −25 ppm になるはずです。

In [ ]:
out = run("proto_analysis.py ii4_msgs.csv")
ppm = [float(x) for x in re.findall(r"時計の進みの差 ([-+][\d.]+) ppm", out)]
bound = [float(x) for x in re.findall(r"片道の誤差の上限の目安 ±([\d.]+) ms", out)]
ok(len(ppm) == 5 and all(abs(p + 25) < 2 for p in ppm), f"時計の進みの差 {ppm} ppm（±数十 ppm 以内。合成の正解は −25 ppm）")
ok(all(b < 10 for b in bound), f"片道の誤差の上限の目安 ±{min(bound)}〜{max(bound)} ms（数 ms）")
rows = {m_.group(1): [float(v) for v in m_.groups()[1:]] for m_ in
        re.finditer(r"^(HTTP|MQTT QoS0|MQTT QoS1|WebSocket)\s+\d+/\d+\s+([\d.]+)\s+\S+\s+\S+\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)", out, re.M)}
for k, v in rows.items():
    print(f"  {k:10s} 欠損率 {v[0]:.2f} %、中央値 {v[1]:.1f} ms、95%点 {v[2]:.1f} ms、平均 {v[4]:.1f} ms")
ok(abs(rows["HTTP"][1] - 21) < 1.5, f"HTTP の遅延の中央値 {rows['HTTP'][1]} ms（本文の合成データの値 約21 ms）")
ok(all(8.5 <= rows[k][1] <= 11.5 for k in ("MQTT QoS0", "MQTT QoS1", "WebSocket")),
   "MQTT QoS0・QoS1・WebSocket の中央値は約 9〜11 ms")
ok(all(rows[k][2] > 2 * rows[k][1] for k in rows), "95%点は、どの方式でも中央値の2倍を超える（右に長い裾）")
ok(rows["MQTT QoS0"][0] > 0 and all(rows[k][0] == 0 for k in ("HTTP", "MQTT QoS1", "WebSocket")),
   "障害がなくても欠けるのは QoS0 だけ（詰まりのときにまとまって欠ける）")
ok("0.00〜0.00" in out, "欠損 0 件の方式では、ブロック単位の区間が「0.00〜0.00」になる（0 だと言えるわけではない。Wilson の上限で言う）")
n_tests = len(re.findall(r"Holm 後", out))
raw, holm = num(r"補正なしで p < 0.05 の数：(\d+)", out), num(r"Holm で補正すると (\d+)", out)
ok(n_tests == 18 and holm < raw, f"18個の比較で、補正なしの「差あり」{raw:.0f} 個のうち Holm の補正で {raw - holm:.0f} 個が消える")
pu, pb = num(r"独立とみなすと p = ([\d.e+-]+)", out), num(r"ブロックの中央値どうしなら p = ([\d.e+-]+)", out)
ok(np.log10(pb) - np.log10(pu) > 10, f"1通ずつ独立とみなした p = {pu:.2g} は、ブロックの中央値どうしの p = {pb:.2g} より何十桁も小さい")
ok(rows["HTTP"][4] > rows["HTTP"][1] and rows["WebSocket"][4] > 1.5 * rows["WebSocket"][1], "平均は少数の大きな遅延に引っぱられ、中央値より大きい")
rtt = {k: num(rf"{k}\s+rtt 中央値\s+([\d.]+) ms", out) for k in ("HTTP", "MQTT QoS0", "MQTT QoS1", "WebSocket")}
ok(rtt["MQTT QoS0"] < 1 and rtt["HTTP"] > rtt["MQTT QoS1"] and rtt["HTTP"] > rtt["WebSocket"],
   "Pico が測った送信の所要時間：QoS0 は 1 ms 未満（書き込みだけ）、HTTP がいちばん長い")
show("proto.png")

## 6. 消費電流を比べる（手順 15、`current_analysis.py`）

確かめること：「5種類のブロックの平均電流、送らないとき（i）との差と1通あたりのエネルギー（mJ）、current.png。電流は数十mA（目安）」。合成の記録の増分は HTTP 3.0 mA、QoS0 1.0、QoS1 1.3、WebSocket 1.1 mA としました（仮の値）。本文の結果の例：「方式による差は数mA 以下で、HTTP がいちばん大きくなりやすい。1通あたり数mJ〜十数mJ」。

In [ ]:
out = run("current_analysis.py data/meter_syn.csv ii4_msgs.csv")
cur = {k: num(rf"{k}\s+\d+ ブロック\s+平均\s+([\d.]+) mA", out) for k in ("HTTP", "送らない", "MQTT QoS0", "MQTT QoS1", "WebSocket")}
ok(all(10 <= v < 100 for v in cur.values()), f"どのブロックの平均電流も数十 mA（{min(cur.values()):.1f}〜{max(cur.values()):.1f} mA）")
mj = {k: num(rf"{k} − 送らない\s+=\s+[-+\d.]+ mA.*1通あたり ([\d.]+) mJ", out) for k in ("HTTP", "MQTT QoS0", "MQTT QoS1", "WebSocket")}
ok(max(mj, key=mj.get) == "HTTP", f"送らないときとの差がいちばん大きいのは HTTP（1通あたり {mj['HTTP']} mJ）")
ok(all(1 <= v <= 20 for v in mj.values()), f"1通あたり {min(mj.values())}〜{max(mj.values())} mJ（数 mJ〜十数 mJ）")
d_http = num(r"HTTP − 送らない\s+=\s+\+([\d.]+) mA", out)
ok(abs(d_http - 3.0) < 0.5, f"HTTP − 送らない = {d_http} mA（合成の正解 3.0 mA）")
show("current.png")

## 7. 障害の時間の流れ（手順 20、`fault_timeline.py`）

確かめること：「方式ごとに、止めていた間の通の数・届いた数・欠損・重複・最大の遅れ・戻してから最初に届くまでの時間が表になり、fault.png ができる。MQTT では LWT が出るまでの秒数（約15秒）も表示される」。

本文 II-4-7 の結果の例（障害 F1）：「QoS0 は、気づくまでの数秒〜十数秒の通が黙って失われる。QoS1・HTTP・WebSocket はバッファから送り直すので欠損は0に近く、そのかわり止めていた間の通の遅れが数十秒〜1分になる」。

In [ ]:
out = run("fault_timeline.py ii4_msgs.csv marks.csv")
tab = {m_.group(1): [float(v) for v in m_.groups()[1:]] for m_ in
       re.finditer(r"^(HTTP|MQTT QoS0|MQTT QoS1|WebSocket)\s+repeater\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)", out, re.M)}
ok(len(tab) == 4, "4つの方式で1回ずつ、中継機を 60 秒抜いた行がある")
ok(tab["MQTT QoS0"][3] > 0, f"QoS0 は止めていた間の {tab['MQTT QoS0'][1]:.0f} 通のうち {tab['MQTT QoS0'][3]:.0f} 通が黙って失われた")
ok(all(tab[k][3] == 0 for k in ("HTTP", "MQTT QoS1", "WebSocket")), "QoS1・HTTP・WebSocket はバッファから送り直すので欠損 0")
ok(all(20 <= tab[k][6] <= 90 for k in ("HTTP", "MQTT QoS1", "WebSocket")),
   "そのかわり最大の遅れは数十秒〜1分程度（" + "、".join(f"{k} {tab[k][6]:.0f} 秒" for k in ("HTTP", "MQTT QoS1", "WebSocket")) + "）")
ok(tab["MQTT QoS1"][4] > 0 or tab["HTTP"][4] > 0, "重複は再送のあとに QoS1・HTTP で出る（WebSocket の合成では入れていない）")
lwt = [float(x) for x in re.findall(r"LWT が出たのは、最後の通が届いてから ([\d.]+) 秒後", out)]
ok(len(lwt) == 2 and all(abs(x - 15) < 1 for x in lwt), f"MQTT では最後の通から {lwt} 秒で LWT（keepalive 10 秒 × 1.5 ＝ 約 15 秒）")
show("fault.png")

## まとめ

- 1・2 節：自作の WebSocket クライアントが RFC 6455 の例どおりに動くこと、1通ごとに接続する HTTP は、つないだままの WebSocket より往復が長く、見出しも大きいことを、計算機の中だけで確かめました。
- 3〜7 節：本文の数値と、合成の記録での遅延・欠損・電流・障害の解析（本文 II-4-7 の合成データの値）を確かめました。合成の値は仮のもので、実機では家の Wi-Fi の混み具合、中継機との距離、ファームの版で大きく変わります。
- このノートブックに入れていないもの：Pi の受け口（Docker・PostgreSQL・Mosquitto）、Pico への書き込みと配線、実物の測定と障害の実験、MQTT のブローカを使った確かめ。これらは教科書の手順 A〜G のとおりに行います。